In [ ]:
# === Google Colab 自動環境セットアップ ===
# ※ローカル環境では無視され、Colab環境でのみ自動でモジュールをインストールします
import sys, os
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/my_PRML'):
        print("リポジトリをダウンロード中...")
        !git clone https://github.com/miyayudai/my_PRML.git > /dev/null 2>&1
    
    print("必要なモジュールをインストール中...")
    %cd /content/my_PRML
    !pip install -q -r requirements.txt
    !pip install -q -e .
    
    print("作業ディレクトリをセットアップ中...")
    %cd /content/my_PRML/13
    print("準備完了！このまま下のセルを実行できます。")

# 第13章 系列データ：演習問題 (Exercises 13.1 - 13.34)

本ノートブックは、PRML（パターン認識と機械学習）第13章「系列データ (Sequential Data)」に登場する**全34問 (Exercises 13.1 〜 13.34)** の詳細な数理解説、証明ステップ、穴埋め問題、および自己完結型の Python 数値検証コードを網羅した完全版です。

### 構成一覧
1. **Exercises 13.1 - 13.4**: マルコフ連鎖、d-分離、未来と過去の条件付き独立性、観測周辺化による大域的相関、高次マルコフ連鎖のマクロ変数表現
2. **Exercises 13.5 - 13.10**: 隠れマルコフモデル (HMM) 最尤推定、Baum-Welch Mステップ更新公式（遷移・初期・ガウス放出・離散放出）、前向き変数 $\alpha$ と後ろ向き変数 $\beta$ の再帰漸化式
3. **Exercises 13.11 - 13.15**: 事後周辺確率 $\gamma(z_n)$ の時間不変正規化、2時点事後確率 $\xi(z_{n-1}, z_n)$、スケーリング係数 $c_n$、ファクターグラフ等価性、自己回帰 HMM
4. **Exercises 13.16 - 13.20**: Viterbi 動的計画法アルゴリズム、Left-to-Right HMM、全探索一致検証、周辺最大系列 vs 最確系列の乖離反例、複数系列学習
5. **Exercises 13.21 - 13.27**: 線形動的システム (LDS)、カルマンゲイン行列の線形ガウス代数導出、ジョセフ形式共分散、RTS カルマンスムーザ、極限解析（観測ノイズ消失・システムノイズ消失）
6. **Exercises 13.28 - 13.34**: LDS の EM パラメータ学習（$A, C, \mathbf{\Gamma}, \mathbf{\Sigma}$）、定常カルマンフィルタと代数的リカッチ方程式 (DARE)、潜在空間の線形変換不変性、Switching LDS、粒子フィルタ (Sequential Importance Resampling)


---
## Exercise 13.1: d-分離による1次マルコフ連鎖の条件付き独立性の証明

### 問題の背景と数学的証明
第13.1節において、1次マルコフ連鎖の結合確率分布は次式のように因数分解される：
$$ p(x_1, \dots, x_N) = p(x_1) \prod_{n=2}^N p(x_n | x_{n-1}) $$
本問では、d-分離規準を用いて条件付き独立性：
$$ p(x_n | x_1, \dots, x_{n-1}) = p(x_n | x_{n-1}) $$
を証明する。

**d-分離による証明**:
1. グラフィカルモデルにおいて、$x_1, \dots, x_{n-2}$ の任意のノード $x_i$ ($i < n-1$) から $x_n$ へのすべての有効パスを考える。
2. このマルコフ連鎖は有向直線グラフ $x_1 \to x_2 \to \dots \to x_{n-1} \to x_n$ であるため、$x_i$ から $x_n$ への唯一のパスは中間ノード $x_{n-1}$ を通過する。
3. このパスにおいて、ノード $x_{n-1}$ は $x_{n-2} \to x_{n-1} \to x_n$ という「head-to-tail」の結合を形成している。
4. d-分離の定義（第8.2節）より、head-to-tail ノードが条件付け集合に含まれる場合、そのノードを通過するパスはブロック（遮断）される。
5. 条件付け集合には $x_{n-1}$ が含まれているため、$x_i$ から $x_n$ へのすべてのパスはブロックされる。
6. したがって、$x_n \perp \{x_1, \dots, x_{n-2}\} \mid x_{n-1}$ が成り立ち、直ちに次式が従う：
$$ p(x_n | x_1, \dots, x_{n-1}) = p(x_n | x_{n-1}) $$

**積の規則による直接計算の確認**:
結合確率の比として評価すると：
$$ p(x_n | x_1, \dots, x_{n-1}) = \frac{p(x_1, \dots, x_n)}{p(x_1, \dots, x_{n-1})} = \frac{p(x_1)\prod_{i=2}^n p(x_i|x_{i-1})}{p(x_1)\prod_{i=2}^{n-1} p(x_i|x_{i-1})} = p(x_n | x_{n-1}) $$
となり、d-分離の結果と完全に一致する。

#### 穴埋め問題
1. 有向グラフ $x_{n-2} \to x_{n-1} \to x_n$ において、ノード $x_{n-1}$ は $\text{[ (A) ]}$ の結合を形成している。
2. 条件付け集合にノード $x_{n-1}$ が含まれるため、パスは $\text{[ (B) ]}$ される。
3. これにより、$x_n$ の分布は直前の $x_{n-1}$ のみに依存し、過去のすべての観測値とは $\text{[ (C) ]}$ となる。
*(解: A: head-to-tail, B: ブロック, C: 条件付き独立)*


In [1]:
# Exercise 13.1 数値検証: 1次マルコフ連鎖における条件付き確率の厳密な等価性
import numpy as np

np.random.seed(42)
K = 3  # 状態数
N = 5  # 系列長

# ランダムな初期確率 pi と推移行列 A
pi = np.random.dirichlet(np.ones(K))
A = np.random.dirichlet(np.ones(K), size=K)

# 全結合確率テーブル p(x_1, ..., x_N) を計算 (K^N 通り)
joint = np.zeros((K, K, K, K, K))
for x1 in range(K):
    for x2 in range(K):
        for x3 in range(K):
            for x4 in range(K):
                for x5 in range(K):
                    joint[x1, x2, x3, x4, x5] = pi[x1] * A[x1, x2] * A[x2, x3] * A[x3, x4] * A[x4, x5]

# p(x_1, x_2, x_3, x_4) を周辺化により算出
p_1234 = np.sum(joint, axis=4)

# 条件付き確率 p(x_5 | x_1, x_2, x_3, x_4) を計算
p_cond = joint / p_1234[:, :, :, :, None]

# 直接の遷移確率 p(x_5 | x_4) と比較
for x1 in range(K):
    for x2 in range(K):
        for x3 in range(K):
            for x4 in range(K):
                np.testing.assert_allclose(p_cond[x1, x2, x3, x4, :], A[x4, :], atol=1e-12)

print("Exercise 13.1 verified: p(x_n | x_1, ..., x_{n-1}) strictly equals p(x_n | x_{n-1}) across all states!")


Exercise 13.1 verified: p(x_n | x_1, ..., x_{n-1}) strictly equals p(x_n | x_{n-1}) across all states!


---
## Exercise 13.2: 現在状態が与えられたときの未来と過去の条件付き独立性

### 問題の背景と数学的証明
マルコフ連鎖において、中間のある時点 $n$ の状態 $x_n$ が観測された場合、未来の系列 $x_{n+1}, \dots, x_N$ と過去の系列 $x_1, \dots, x_{n-1}$ は条件付き独立：
$$ p(x_1, \dots, x_{n-1}, x_{n+1}, \dots, x_N | x_n) = p(x_1, \dots, x_{n-1} | x_n) p(x_{n+1}, \dots, x_N | x_n) $$
が成立することを証明する。

**代数的証明**:
1. 条件付き確率の定義と因数分解公式より：
$$ p(x_1, \dots, x_N | x_n) = \frac{p(x_1, \dots, x_N)}{p(x_n)} $$
2. 分子の結合分布を $x_n$ を境に分割する：
$$ p(x_1, \dots, x_N) = \left( p(x_1) \prod_{i=2}^n p(x_i | x_{i-1}) \right) \left( \prod_{j=n+1}^N p(x_j | x_{j-1}) \right) = p(x_1, \dots, x_n) p(x_{n+1}, \dots, x_N | x_n) $$
3. これを分母 $p(x_n)$ で割ると：
$$ \frac{p(x_1, \dots, x_N)}{p(x_n)} = \frac{p(x_1, \dots, x_n)}{p(x_n)} p(x_{n+1}, \dots, x_N | x_n) = p(x_1, \dots, x_{n-1} | x_n) p(x_{n+1}, \dots, x_N | x_n) $$
4. したがって、$x_n$ で条件付けると、過去と未来は互いに因数分解され、完全に独立となる。

#### 穴埋め問題
1. マルコフ性により、$x_n$ が与えられたとき、未来の観測値の生成は $\text{[ (A) ]}$ の状態のみに依存する。
2. 過去と未来の結合条件付き確率は、過去の条件付き確率と未来の条件付き確率の $\text{[ (B) ]}$ に分解される。
3. したがって、$x_n$ は過去と未来を $\text{[ (C) ]}$ する。
*(解: A: 現在 ($x_n$), B: 積, C: 条件付き独立化)*


In [2]:
# Exercise 13.2 数値検証: 現在状態を与えたときの過去と未来の独立性
p_123 = np.sum(joint, axis=(3, 4))  # (x1, x2, x3)
p_2 = np.sum(p_123, axis=(0, 2))    # (x2,)
p_12 = np.sum(p_123, axis=2)        # (x1, x2)
p_23 = np.sum(p_123, axis=0)        # (x2, x3)

for x2 in range(K):
    p_13_given_2 = p_123[:, x2, :] / p_2[x2]
    p_1_given_2 = p_12[:, x2] / p_2[x2]
    p_3_given_2 = p_23[x2, :] / p_2[x2]
    p_prod = np.outer(p_1_given_2, p_3_given_2)
    
    np.testing.assert_allclose(p_13_given_2, p_prod, atol=1e-12)

print("Exercise 13.2 verified: Past and Future are strictly conditionally independent given the present state!")


Exercise 13.2 verified: Past and Future are strictly conditionally independent given the present state!


---
## Exercise 13.3: 潜在変数を周辺化した場合の観測変数間大域的相関

### 問題の背景と数学的証明
隠れマルコフモデル (HMM) では、観測変数系列 $X = \{x_1, \dots, x_N\}$ と潜在変数系列 $Z = \{z_1, \dots, z_N\}$ の結合分布は：
$$ p(X, Z) = p(z_1) p(x_1 | z_1) \prod_{n=2}^N p(z_n | z_{n-1}) p(x_n | z_n) $$
で与えられる。
潜在変数 $Z$ が観測されず周辺化されたとき、$X$ の周辺分布 $p(X) = \sum_Z p(X, Z)$ は因数分解せず、すべての観測変数間に大域的な相関が生じることを d-分離を用いて証明する。

**d-分離による証明**:
1. 任意の2つの観測ノード $x_i$ と $x_j$ ($i < j$) を結ぶパスを考える：
$$ x_i \leftarrow z_i \to z_{i+1} \to \dots \to z_j \to x_j $$
2. パス上の中間ノード $z_i$ は tail-to-tail、その後の $z_k$ ($i < k < j$) は head-to-tail の構造を持つ。
3. 潜在変数 $Z$ はすべて未観測（条件付け集合に含まれない）である。
4. d-分離の規則：
   - head-to-tail ノードは未観測のときブロック**されない**。
   - tail-to-tail ノードは未観測のときブロック**されない**。
5. したがって、$x_i$ と $x_j$ を結ぶパス上にはブロックするノードが1つも存在せず、パスは常に「オープン」である。
6. よって、任意の $i \neq j$ について $x_i \not\perp x_j$ であり、観測変数間は一般に独立ではない（大域的相関を持つ）。

#### 穴埋め問題
1. $z_i \to z_{i+1}$ の構造において、未観測のノードはパスを $\text{[ (A) ]}$。
2. 潜在変数を周辺化すると、任意の観測変数 $x_i$ と $x_j$ の間のパスは $\text{[ (B) ]}$ となる。
3. この結果、観測系列は完全な $\text{[ (C) ]}$ を獲得し、長距離依存性を表現できる。
*(解: A: ブロックしない, B: オープン (非ブロック), C: 大域的相関)*


In [3]:
# Exercise 13.3 数値検証: 潜在変数を周辺化した場合の観測変数間相互情報量 I(X_1; X_3) > 0
pi_h = np.array([0.5, 0.5])
A_h = np.array([[0.8, 0.2],
                [0.2, 0.8]])
B_h = np.array([[0.9, 0.1],
                [0.1, 0.9]])

p_full = np.zeros((2, 2, 2, 2, 2, 2))
for z1 in range(2):
    for z2 in range(2):
        for z3 in range(2):
            for x1 in range(2):
                for x2 in range(2):
                    for x3 in range(2):
                        p_full[x1, x2, x3, z1, z2, z3] = (
                            pi_h[z1] * B_h[z1, x1] *
                            A_h[z1, z2] * B_h[z2, x2] *
                            A_h[z2, z3] * B_h[z3, x3]
                        )

p_obs = np.sum(p_full, axis=(3, 4, 5))
p_x1_x3 = np.sum(p_obs, axis=1)
p_x1 = np.sum(p_x1_x3, axis=1)
p_x3 = np.sum(p_x1_x3, axis=0)

mi_x1_x3 = np.sum(p_x1_x3 * np.log(p_x1_x3 / np.outer(p_x1, p_x3)))

print(f"Mutual Information I(X_1; X_3) after marginalizing latent states: {mi_x1_x3:.6f}")
assert mi_x1_x3 > 1e-4, "X_1 and X_3 must be correlated (I(X_1; X_3) > 0)!"

p_x1_x3_z2 = np.sum(p_full, axis=(1, 3, 5))
p_z2 = np.sum(p_x1_x3_z2, axis=(0, 1))
for z2 in range(2):
    p_cond_x1_x3 = p_x1_x3_z2[:, :, z2] / p_z2[z2]
    p_cond_x1 = np.sum(p_cond_x1_x3, axis=1)
    p_cond_x3 = np.sum(p_cond_x1_x3, axis=0)
    np.testing.assert_allclose(p_cond_x1_x3, np.outer(p_cond_x1, p_cond_x3), atol=1e-12)

print("Exercise 13.3 verified: Unobserved latents induce global correlations, while conditioning d-separates!")


Mutual Information I(X_1; X_3) after marginalizing latent states: 0.026782
Exercise 13.3 verified: Unobserved latents induce global correlations, while conditioning d-separates!


---
## Exercise 13.4: 高次マルコフ連鎖のマクロ変数結合による1次マルコフ連鎖への帰着

### 問題の背景と数学的証明
$M$ 次マルコフ連鎖は、各観測値 $x_n$ の条件付き確率が直前の $M$ 個の観測値に依存するモデルである：
$$ p(x_1, \dots, x_N) = p(x_1, \dots, x_M) \prod_{n=M+1}^N p(x_n | x_{n-1}, \dots, x_{n-M}) $$
本問では、連続する $M$ 個の変数をまとめた新しい複合（マクロ）変数 $\mathbf{y}_n = (x_n, x_{n-1}, \dots, x_{n-M+1})^{\mathrm{T}}$ を定義することで、任意の $M$ 次マルコフ連鎖がマクロ変数に関する標準的な「1次マルコフ連鎖」として厳密に書き直せることを証明する。

**代数的証明**:
1. 新変数 $\mathbf{y}_n = (x_n, \dots, x_{n-M+1})$ と $\mathbf{y}_{n-1} = (x_{n-1}, \dots, x_{n-M})$ を考える。
2. 条件付き確率 $p(\mathbf{y}_n | \mathbf{y}_{n-1})$ は次のように定義される：
$$ p(\mathbf{y}_n | \mathbf{y}_{n-1}) = p(x_n, x_{n-1}', \dots, x_{n-M+1}' | x_{n-1}, \dots, x_{n-M}) $$
3. ここで、要素の整合性制約から $x_{n-1}' = x_{n-1}, \dots, x_{n-M+1}' = x_{n-M+1}$ が決定論的に満たされる必要があるため：
$$ p(\mathbf{y}_n | \mathbf{y}_{n-1}) = p(x_n | x_{n-1}, \dots, x_{n-M}) \prod_{i=1}^{M-1} \mathbb{I}(x_{n-i}' = x_{n-i}) $$
4. これにより、結合確率は1次マルコフ連鎖の形式：
$$ p(\mathbf{y}_M, \dots, \mathbf{y}_N) = p(\mathbf{y}_M) \prod_{n=M+1}^N p(\mathbf{y}_n | \mathbf{y}_{n-1}) $$
として完全に因数分解される。
5. 各変数が $K$ 個の離散状態をとる場合、新変数 $\mathbf{y}_n$ の状態数は $K^M$ となる。したがって、高次マルコフ連鎖は状態数が指数的に拡大した1次マルコフ連鎖と等価である。

#### 穴埋め問題
1. $M$ 次マルコフ連鎖を1次マルコフ連鎖に書き換えた場合、新しい状態空間のサイズは $\text{[ (A) ]}$ となる。
2. 遷移確率行列において、過去のシフトが一致しない要素の遷移確率は $\text{[ (B) ]}$ である。
3. したがって、高次連鎖は1次連鎖の $\text{[ (C) ]}$ 遷移行列モデルとして包含される。
*(解: A: $K^M$, B: $0$, C: 希薄 (スパース))*


In [4]:
# Exercise 13.4 数値検証: 2次マルコフ連鎖 (M=2, K=2) の1次マルコフ連鎖表現
K = 2
M = 2
P_2nd = np.random.dirichlet(np.ones(K), size=(K, K))

K_macro = K ** M
T_macro = np.zeros((K_macro, K_macro))

for y_prev in range(K_macro):
    x_prev2 = y_prev // K
    x_prev1 = y_prev % K
    for y_curr in range(K_macro):
        x_curr1 = y_curr // K
        x_curr0 = y_curr % K
        if x_curr1 == x_prev1:
            T_macro[y_prev, y_curr] = P_2nd[x_prev2, x_prev1, x_curr0]
        else:
            T_macro[y_prev, y_curr] = 0.0

np.testing.assert_allclose(np.sum(T_macro, axis=1), np.ones(K_macro), atol=1e-12)

seq = [0, 1, 0, 0, 1, 1, 0]
ll_2nd = sum(np.log(P_2nd[seq[i-2], seq[i-1], seq[i]]) for i in range(2, len(seq)))
macro_seq = [seq[i-1]*K + seq[i] for i in range(1, len(seq))]
ll_macro = sum(np.log(T_macro[macro_seq[j-1], macro_seq[j]]) for j in range(1, len(macro_seq)))

np.testing.assert_allclose(ll_2nd, ll_macro, atol=1e-12)
print(f"Exercise 13.4 verified: Log-likelihoods match identically ({ll_2nd:.6f} == {ll_macro:.6f})!")


Exercise 13.4 verified: Log-likelihoods match identically (-5.115347 == -5.115347)!


---
## Exercise 13.5 ⭐: Baum-Welch M-step における遷移確率行列 $A_{jk}$ のラグランジュ乗数法による更新式導出

### 問題の背景と数学的証明
Baum-Welch (EM) アルゴリズムの E ステップにおいて、現在のパラメータ $\boldsymbol{\theta}^{\text{old}}$ の下での完全データ対数尤度の期待値 $Q(\boldsymbol{\theta}, \boldsymbol{\theta}^{\text{old}})$ は次式で与えられる（(13.17)式）：
$$ Q(\boldsymbol{\theta}, \boldsymbol{\theta}^{\text{old}}) = \sum_{k=1}^K \gamma(z_{1k}) \ln \pi_k + \sum_{n=2}^N \sum_{j=1}^K \sum_{k=1}^K \xi(z_{n-1,j}, z_{nk}) \ln A_{jk} + \sum_{n=1}^N \sum_{k=1}^K \gamma(z_{nk}) \ln p(\mathbf{x}_n | \boldsymbol{\phi}_k) $$
ここで $\xi(z_{n-1,j}, z_{nk}) = p(z_{n-1,j}=1, z_{nk}=1 | \mathbf{X}, \boldsymbol{\theta}^{\text{old}})$ である。
M ステップでは、各行の確率の和が1であるという制約：
$$ \sum_{k=1}^K A_{jk} = 1 \quad (j = 1, \dots, K) $$
の下で、$Q(\boldsymbol{\theta}, \boldsymbol{\theta}^{\text{old}})$ を $A_{jk}$ に関して最大化する。

**ラグランジュ未定乗数法による導出**:
1. 各 $j$ に対する制約条件を取り入れたラグランジュ関数 $L(\{A_{jk}\}, \{\lambda_j\})$ を定義する：
$$ L = \sum_{n=2}^N \sum_{j=1}^K \sum_{k=1}^K \xi(z_{n-1,j}, z_{nk}) \ln A_{jk} + \sum_{j=1}^K \lambda_j \left( 1 - \sum_{k=1}^K A_{jk} \right) $$
2. $A_{jk}$ に関して偏微分してゼロとおく：
$$ \frac{\partial L}{\partial A_{jk}} = \frac{\sum_{n=2}^N \xi(z_{n-1,j}, z_{nk})}{A_{jk}} - \lambda_j = 0 $$
3. したがって：
$$ A_{jk} = \frac{1}{\lambda_j} \sum_{n=2}^N \xi(z_{n-1,j}, z_{nk}) $$
4. 制約 $\sum_{k=1}^K A_{jk} = 1$ に代入して未定乗数 $\lambda_j$ を決定する：
$$ \sum_{k=1}^K A_{jk} = \frac{1}{\lambda_j} \sum_{n=2}^N \sum_{k=1}^K \xi(z_{n-1,j}, z_{nk}) = 1 \implies \lambda_j = \sum_{n=2}^N \sum_{k=1}^K \xi(z_{n-1,j}, z_{nk}) $$
5. ここで、事後結合確率の定義から $\sum_{k=1}^K \xi(z_{n-1,j}, z_{nk}) = \gamma(z_{n-1,j})$ であるため：
$$ \lambda_j = \sum_{n=2}^N \gamma(z_{n-1,j}) $$
6. これを代入すると、求める Baum-Welch M-step 更新公式が得られる：
$$ A_{jk} = \frac{\sum_{n=2}^N \xi(z_{n-1,j}, z_{nk})}{\sum_{n=2}^N \gamma(z_{n-1,j})} $$

#### 穴埋め問題
1. 遷移確率行列の各行は確率分布であるため、制約条件は $\text{[ (A) ]} = 1$ である。
2. ラグランジュ関数を $A_{jk}$ で偏微分すると、$A_{jk} \propto \text{[ (B) ]}$ となる。
3. 周辺化関係 $\sum_{k=1}^K \xi(z_{n-1,j}, z_{nk}) = \text{[ (C) ]}$ を用いることで、分母が簡単化される。
*(解: A: $\sum_{k=1}^K A_{jk}$, B: $\sum_{n=2}^N \xi(z_{n-1,j}, z_{nk})$, C: $\gamma(z_{n-1,j})$)*


In [5]:
# Exercise 13.5 数値検証: Baum-Welch Mステップ更新公式と数値的制約最適化の一致
import numpy as np
from scipy.optimize import minimize

np.random.seed(42)
N = 10
K = 3

# モックの事後統計量 xi (N-1, K, K)
xi = np.random.dirichlet(np.ones(K*K), size=N-1).reshape(N-1, K, K)
gamma = np.sum(xi, axis=2)  # (N-1, K)

# 解析公式による A_jk の計算
xi_sum = np.sum(xi, axis=0)  # (K, K)
gamma_sum = np.sum(gamma, axis=0)  # (K,)
A_analytical = xi_sum / gamma_sum[:, None]

# 数値的最適化 (SLSQP) による解
A_numerical = np.zeros((K, K))
for j in range(K):
    def objective(a_row):
        # 負の対数期待尤度
        return -np.sum(xi_sum[j, :] * np.log(a_row + 1e-15))
    
    cons = ({'type': 'eq', 'fun': lambda a: np.sum(a) - 1.0})
    bounds = [(1e-6, 1.0) for _ in range(K)]
    init = np.ones(K) / K
    res = minimize(objective, init, method='SLSQP', bounds=bounds, constraints=cons, options={'ftol': 1e-10, 'maxiter': 200})
    A_numerical[j, :] = res.x

np.testing.assert_allclose(A_analytical, A_numerical, atol=1e-3)
np.testing.assert_allclose(np.sum(A_analytical, axis=1), np.ones(K), atol=1e-12)
print("Exercise 13.5 verified: Analytical Baum-Welch M-step matches constrained numerical optimization perfectly!")


Exercise 13.5 verified: Analytical Baum-Welch M-step matches constrained numerical optimization perfectly!


---
## Exercise 13.6: 初期状態確率ベクトル $\pi_k$ の制約付き最尤推定公式の導出

### 問題の背景と数学的証明
初期潜在状態 $z_1$ の事前確率ベクトル $\boldsymbol{\pi} = (\pi_1, \dots, \pi_K)^{\mathrm{T}}$ は $\sum_{k=1}^K \pi_k = 1$ を満たす。
$Q(\boldsymbol{\theta}, \boldsymbol{\theta}^{\text{old}})$ の中で $\boldsymbol{\pi}$ に依存する項は第1項のみである：
$$ Q_\pi = \sum_{k=1}^K \gamma(z_{1k}) \ln \pi_k $$
ここで $\gamma(z_{1k}) = p(z_{1k}=1 | \mathbf{X}, \boldsymbol{\theta}^{\text{old}})$ である。

**ラグランジュ乗数法による導出**:
1. ラグランジュ関数：
$$ L(\boldsymbol{\pi}, \lambda) = \sum_{k=1}^K \gamma(z_{1k}) \ln \pi_k + \lambda \left( 1 - \sum_{k=1}^K \pi_k \right) $$
2. $\pi_k$ で微分してゼロとおく：
$$ \frac{\partial L}{\partial \pi_k} = \frac{\gamma(z_{1k})}{\pi_k} - \lambda = 0 \implies \pi_k = \frac{\gamma(z_{1k})}{\lambda} $$
3. 制約 $\sum_k \pi_k = 1$ より $\lambda = \sum_{k=1}^K \gamma(z_{1k})$。
4. 事後周辺確率の正規化条件 $\sum_{k=1}^K \gamma(z_{1k}) = 1$ より、$\lambda = 1$ となる。
5. したがって、更新公式は：
$$ \pi_k = \gamma(z_{1k}) $$
となる。

#### 穴埋め問題
1. $\pi_k$ に関するラグランジュ乗数は、事後確率の総和より $\lambda = \text{[ (A) ]}$ となる。
2. したがって、M ステップにおける最適初期確率は、時刻 $1$ における $\text{[ (B) ]}$ にそのまま一致する。
*(解: A: $1$, B: 事後周辺確率 $\gamma(z_{1k})$)*


In [6]:
# Exercise 13.6 数値検証: 初期確率 pi_k の更新式
gamma_1 = np.array([0.2, 0.5, 0.3])
# 解析解
pi_opt = gamma_1.copy()

# 数値最適化による確認
def obj_pi(pi):
    return -np.sum(gamma_1 * np.log(pi + 1e-15))

cons_pi = ({'type': 'eq', 'fun': lambda p: np.sum(p) - 1.0})
res_pi = minimize(obj_pi, np.ones(3)/3, method='SLSQP', bounds=[(1e-6, 1.0)]*3, constraints=cons_pi, options={'ftol': 1e-10, 'maxiter': 200})

np.testing.assert_allclose(pi_opt, res_pi.x, atol=1e-3)
print(f"Exercise 13.6 verified: pi_k = gamma(z_1k) correctly maximizes Q_pi: {pi_opt}")


Exercise 13.6 verified: pi_k = gamma(z_1k) correctly maximizes Q_pi: [0.2 0.5 0.3]


---
## Exercise 13.7: 多変量ガウス放出モデルにおける平均 $\boldsymbol{\mu}_k$ と共分散 $\mathbf{\Sigma}_k$ の M-step 更新式の導出

### 問題の背景と数学的証明
連続観測変数 $\mathbf{x}_n \in \mathbb{R}^D$ に対する放出分布が多変量正規分布 $p(\mathbf{x}_n | z_k) = \mathcal{N}(\mathbf{x}_n | \boldsymbol{\mu}_k, \mathbf{\Sigma}_k)$ の場合、
$Q$ 関数の放出確率項は：
$$ Q_\phi = -\frac{1}{2} \sum_{n=1}^N \sum_{k=1}^K \gamma(z_{nk}) \left( \ln |\mathbf{\Sigma}_k| + (\mathbf{x}_n - \boldsymbol{\mu}_k)^{\mathrm{T}} \mathbf{\Sigma}_k^{-1} (\mathbf{x}_n - \boldsymbol{\mu}_k) + D \ln(2\pi) \right) $$
となる。

**パラメータ導出**:
1. **平均 $\boldsymbol{\mu}_k$ の最大化**:
$$ \frac{\partial Q_\phi}{\partial \boldsymbol{\mu}_k} = \sum_{n=1}^N \gamma(z_{nk}) \mathbf{\Sigma}_k^{-1} (\mathbf{x}_n - \boldsymbol{\mu}_k) = \mathbf{0} $$
両辺に左から $\mathbf{\Sigma}_k$ を掛けると：
$$ \sum_{n=1}^N \gamma(z_{nk}) (\mathbf{x}_n - \boldsymbol{\mu}_k) = \mathbf{0} \implies \boldsymbol{\mu}_k = \frac{\sum_{n=1}^N \gamma(z_{nk}) \mathbf{x}_n}{\sum_{n=1}^N \gamma(z_{nk})} $$
2. **共分散 $\mathbf{\Sigma}_k$ の最大化**:
精度行列 $\mathbf{\Lambda}_k = \mathbf{\Sigma}_k^{-1}$ に関して微分する：
$$ \frac{\partial Q_\phi}{\partial \mathbf{\Lambda}_k} = \frac{1}{2} \sum_{n=1}^N \gamma(z_{nk}) \left( \mathbf{\Lambda}_k^{-1} - (\mathbf{x}_n - \boldsymbol{\mu}_k)(\mathbf{x}_n - \boldsymbol{\mu}_k)^{\mathrm{T}} \right) = \mathbf{0} $$
したがって：
$$ \mathbf{\Sigma}_k = \frac{\sum_{n=1}^N \gamma(z_{nk}) (\mathbf{x}_n - \boldsymbol{\mu}_k)(\mathbf{x}_n - \boldsymbol{\mu}_k)^{\mathrm{T}}}{\sum_{n=1}^N \gamma(z_{nk})} $$
これは負担率 $\gamma(z_{nk})$ を重みとする重み付き最尤推定量に一致する。

#### 穴埋め問題
1. $\boldsymbol{\mu}_k$ の推定量は、観測データ $\mathbf{x}_n$ を負担率 $\gamma(z_{nk})$ で重み付けした $\text{[ (A) ]}$ である。
2. 共分散行列 $\mathbf{\Sigma}_k$ の推定量は、中心化二乗誤差の $\text{[ (B) ]}$ となる。
*(解: A: 重み付き平均, B: 重み付き標本共分散)*


In [7]:
# Exercise 13.7 数値検証: ガウス放出モデルの重み付き最尤推定解
D = 2
N = 20
X_data = np.random.randn(N, D)
gamma_k = np.random.uniform(0.1, 0.9, size=N)

# 解析公式による計算
N_k = np.sum(gamma_k)
mu_k_analytical = np.sum(gamma_k[:, None] * X_data, axis=0) / N_k
diff = X_data - mu_k_analytical
Sigma_k_analytical = (gamma_k[:, None, None] * (diff[:, :, None] @ diff[:, None, :])).sum(axis=0) / N_k

# 勾配が厳密にゼロであることを確認
grad_mu = np.sum(gamma_k[:, None] * np.linalg.solve(Sigma_k_analytical, (X_data - mu_k_analytical).T).T, axis=0)
np.testing.assert_allclose(grad_mu, np.zeros(D), atol=1e-10)

print(f"Exercise 13.7 verified: Gradient at analytical mu_k is zero: {grad_mu}")
print(f"Sigma_k condition number: {np.linalg.cond(Sigma_k_analytical):.4f}")


Exercise 13.7 verified: Gradient at analytical mu_k is zero: [2.22044605e-16 1.66533454e-16]
Sigma_k condition number: 6.0420


---
## Exercise 13.8: 離散多項放出モデルにおける放出確率行列 $\phi_{km}$ の M-step 更新式の導出

### 問題の背景と数学的証明
観測変数 $x_n$ が $M$ 種類の離散シンボル $x_n \in \{1, \dots, M\}$ をとる場合、放出確率は行列 $\phi_{km} = p(x_n = m | z_{nk} = 1)$ で表現され、各状態 $k$ について $\sum_{m=1}^M \phi_{km} = 1$ を満たす。
$Q$ 関数の放出項は：
$$ Q_\phi = \sum_{n=1}^N \sum_{k=1}^K \sum_{m=1}^M \gamma(z_{nk}) \mathbb{I}(x_n = m) \ln \phi_{km} $$
である。

**ラグランジュ乗数法による導出**:
1. 各 $k$ の制約に対するラグランジュ関数：
$$ L = \sum_{n=1}^N \sum_{k=1}^K \sum_{m=1}^M \gamma(z_{nk}) \mathbb{I}(x_n = m) \ln \phi_{km} + \sum_{k=1}^K \lambda_k \left( 1 - \sum_{m=1}^M \phi_{km} \right) $$
2. $\phi_{km}$ で微分してゼロとおく：
$$ \frac{\partial L}{\partial \phi_{km}} = \frac{\sum_{n=1}^N \gamma(z_{nk}) \mathbb{I}(x_n = m)}{\phi_{km}} - \lambda_k = 0 \implies \phi_{km} = \frac{\sum_{n=1}^N \gamma(z_{nk}) \mathbb{I}(x_n = m)}{\lambda_k} $$
3. 制約 $\sum_{m=1}^M \phi_{km} = 1$ より：
$$ \lambda_k = \sum_{m=1}^M \sum_{n=1}^N \gamma(z_{nk}) \mathbb{I}(x_n = m) = \sum_{n=1}^N \gamma(z_{nk}) $$
4. したがって、求める M-step 更新式は：
$$ \phi_{km} = \frac{\sum_{n=1}^N \gamma(z_{nk}) \mathbb{I}(x_n = m)}{\sum_{n=1}^N \gamma(z_{nk})} $$
となる。

#### 穴埋め問題
1. 離散放出モデルでは、シンボル $m$ が観測された時刻の負担率の和を、全時刻の負担率の和で $\text{[ (A) ]}$ する。
2. これにより得られた推定量は、各状態における離散分布の $\text{[ (B) ]}$ に一致する。
*(解: A: 除算 (正規化), B: 最尤推定量)*


In [8]:
# Exercise 13.8 数値検証: 離散多項放出モデルの M ステップ更新
N_samples = 30
M_symbols = 4
K_states = 2
obs = np.random.choice(M_symbols, size=N_samples)
gamma_mat = np.random.dirichlet(np.ones(K_states), size=N_samples) # (N, K)

phi_analytical = np.zeros((K_states, M_symbols))
for k in range(K_states):
    for m in range(M_symbols):
        phi_analytical[k, m] = np.sum(gamma_mat[obs == m, k]) / np.sum(gamma_mat[:, k])

# 行和が 1 であることを検証
np.testing.assert_allclose(np.sum(phi_analytical, axis=1), np.ones(K_states), atol=1e-12)
print("Exercise 13.8 verified: Discrete emission probabilities sum strictly to 1 across all states:")
print(phi_analytical)


Exercise 13.8 verified: Discrete emission probabilities sum strictly to 1 across all states:
[[0.18838633 0.335112   0.17827351 0.29822816]
 [0.21316812 0.18906014 0.29576274 0.30200899]]


---
## Exercise 13.9: Forward-Backward アルゴリズム：前向き変数 $\alpha(z_n)$ の再帰漸化式の導出

### 問題の背景と数学的証明
前向き変数 $\alpha(z_n)$ は、時刻 $n$ までの観測系列と時刻 $n$ の潜在状態の結合確率として定義される（(13.34)式）：
$$ \alpha(z_n) \equiv p(\mathbf{x}_1, \dots, \mathbf{x}_n, z_n) $$
本問では、条件付き独立性を利用して、前向き再帰漸化式（(13.36)式）：
$$ \alpha(z_n) = p(\mathbf{x}_n | z_n) \sum_{z_{n-1}} \alpha(z_{n-1}) p(z_n | z_{n-1}) $$
を代数的に導出する。

**代数的導出**:
1. 定義より、結合確率を積の規則で展開する：
$$ p(\mathbf{x}_1, \dots, \mathbf{x}_n, z_n) = p(\mathbf{x}_n | \mathbf{x}_1, \dots, \mathbf{x}_{n-1}, z_n) p(\mathbf{x}_1, \dots, \mathbf{x}_{n-1}, z_n) $$
2. HMM の条件付き独立性より、$\mathbf{x}_n$ は $z_n$ のみが与えられれば過去の観測値とは独立であるため：
$$ p(\mathbf{x}_n | \mathbf{x}_1, \dots, \mathbf{x}_{n-1}, z_n) = p(\mathbf{x}_n | z_n) $$
3. 第2項 $p(\mathbf{x}_1, \dots, \mathbf{x}_{n-1}, z_n)$ に直前の潜在状態 $z_{n-1}$ を導入し、周辺化の和をとる：
$$ p(\mathbf{x}_1, \dots, \mathbf{x}_{n-1}, z_n) = \sum_{z_{n-1}} p(\mathbf{x}_1, \dots, \mathbf{x}_{n-1}, z_{n-1}, z_n) $$
4. ここで結合確率を条件付き確率に分解する：
$$ p(\mathbf{x}_1, \dots, \mathbf{x}_{n-1}, z_{n-1}, z_n) = p(z_n | \mathbf{x}_1, \dots, \mathbf{x}_{n-1}, z_{n-1}) p(\mathbf{x}_1, \dots, \mathbf{x}_{n-1}, z_{n-1}) $$
5. マルコフ性より $p(z_n | \mathbf{x}_1, \dots, \mathbf{x}_{n-1}, z_{n-1}) = p(z_n | z_{n-1})$、また定義より $p(\mathbf{x}_1, \dots, \mathbf{x}_{n-1}, z_{n-1}) = \alpha(z_{n-1})$ である。
6. 以上をまとめると：
$$ \alpha(z_n) = p(\mathbf{x}_n | z_n) \sum_{z_{n-1}} \alpha(z_{n-1}) p(z_n | z_{n-1}) $$
初期条件は $\alpha(z_1) = p(\mathbf{x}_1, z_1) = p(z_1) p(\mathbf{x}_1 | z_1) = \pi_k p(\mathbf{x}_1 | \boldsymbol{\phi}_k)$ である。

#### 穴埋め問題
1. $\mathbf{x}_n$ の放出確率は、時刻 $n$ の潜在状態 $z_n$ のみに依存するため $\text{[ (A) ]}$ と簡略化される。
2. 直前状態の同時確率は、前向き変数 $\alpha(z_{n-1})$ と遷移行列 $A_{j k} = \text{[ (B) ]}$ の積の和となる。
3. したがって、各ステップの計算量は状態数 $K$ に対し $\text{[ (C) ]}$ で実行できる。
*(解: A: $p(\mathbf{x}_n | z_n)$, B: $p(z_n | z_{n-1})$, C: $\mathcal{O}(K^2)$)*


In [9]:
# Exercise 13.9 数値検証: 前向き再帰法と直接結合確率周辺化の一致
N_steps = 4
K_states = 2
pi_vec = np.array([0.6, 0.4])
A_mat = np.array([[0.7, 0.3],
                  [0.4, 0.6]])
B_mat = np.array([[0.8, 0.2],
                  [0.1, 0.9]])
obs_seq = [0, 1, 0, 1]

# 1. 前向きアルゴリズムによる alpha の計算
alpha = np.zeros((N_steps, K_states))
alpha[0, :] = pi_vec * B_mat[:, obs_seq[0]]
for n in range(1, N_steps):
    alpha[n, :] = B_mat[:, obs_seq[n]] * (alpha[n-1, :] @ A_mat)

# 2. ブルートフォース全状態同時確率計算による alpha(z_N) の算出
# p(x_1, ..., x_N, z_N) = sum_{z_1...z_{N-1}} p(x_1...x_N, z_1...z_N)
alpha_brute = np.zeros(K_states)
for z1 in range(K_states):
    for z2 in range(K_states):
        for z3 in range(K_states):
            for z4 in range(K_states):
                p_joint = (pi_vec[z1] * B_mat[z1, obs_seq[0]] *
                           A_mat[z1, z2] * B_mat[z2, obs_seq[1]] *
                           A_mat[z2, z3] * B_mat[z3, obs_seq[2]] *
                           A_mat[z3, z4] * B_mat[z4, obs_seq[3]])
                alpha_brute[z4] += p_joint

np.testing.assert_allclose(alpha[-1, :], alpha_brute, atol=1e-12)
print("Exercise 13.9 verified: Forward recursion alpha strictly matches full brute force marginalization!")


Exercise 13.9 verified: Forward recursion alpha strictly matches full brute force marginalization!


---
## Exercise 13.10: Forward-Backward アルゴリズム：後ろ向き変数 $\beta(z_n)$ の再帰漸化式の導出

### 問題の背景と数学的証明
後ろ向き変数 $\beta(z_n)$ は、現在の潜在状態 $z_n$ が与えられたときの未来の観測系列の条件付き確率として定義される（(13.44)式）：
$$ \beta(z_n) \equiv p(\mathbf{x}_{n+1}, \dots, \mathbf{x}_N | z_n) $$
本問では、後ろ向き再帰漸化式（(13.47)式）：
$$ \beta(z_n) = \sum_{z_{n+1}} \beta(z_{n+1}) p(\mathbf{x}_{n+1} | z_{n+1}) p(z_{n+1} | z_n) $$
および終端境界条件 $\beta(z_N) = 1$ を代数的に証明する。

**代数的導出**:
1. 条件付き確率の定義より、未来の潜在状態 $z_{n+1}$ を導入して周辺化する：
$$ p(\mathbf{x}_{n+1}, \dots, \mathbf{x}_N | z_n) = \sum_{z_{n+1}} p(\mathbf{x}_{n+1}, \dots, \mathbf{x}_N, z_{n+1} | z_n) $$
2. 被和数を積の規則で展開する：
$$ p(\mathbf{x}_{n+1}, \dots, \mathbf{x}_N, z_{n+1} | z_n) = p(\mathbf{x}_{n+2}, \dots, \mathbf{x}_N | \mathbf{x}_{n+1}, z_{n+1}, z_n) p(\mathbf{x}_{n+1} | z_{n+1}, z_n) p(z_{n+1} | z_n) $$
3. d-分離と条件付き独立性より：
   - $z_{n+1}$ が与えられれば、未来の観測 $\mathbf{x}_{n+2}, \dots, \mathbf{x}_N$ は $\mathbf{x}_{n+1}$ および $z_n$ と独立である：$p(\mathbf{x}_{n+2}, \dots, \mathbf{x}_N | z_{n+1}) = \beta(z_{n+1})$
   - $z_{n+1}$ が与えられれば、$\mathbf{x}_{n+1}$ は $z_n$ と独立である：$p(\mathbf{x}_{n+1} | z_{n+1}, z_n) = p(\mathbf{x}_{n+1} | z_{n+1})$
4. これらを代入すると：
$$ \beta(z_n) = \sum_{z_{n+1}} \beta(z_{n+1}) p(\mathbf{x}_{n+1} | z_{n+1}) p(z_{n+1} | z_n) $$
5. 終端時刻 $N$ においては、未来の観測値が存在しない空集合であるため：
$$ p(\emptyset | z_N) = 1 \implies \beta(z_N) = 1 $$
が自然に満たされる。

#### 穴埋め問題
1. 後ろ向き変数 $\beta(z_n)$ は、未来の観測値が与えられた条件付き確率ではなく、状態 $z_n$ が与えられたときの未来の観測値の $\text{[ (A) ]}$ である。
2. 漸化式において、未来側からのメッセージに $\text{[ (B) ]}$ と遷移確率を掛けて状態 $z_{n+1}$ について和をとる。
3. 終端境界条件は $\beta(z_N) = \text{[ (C) ]}$ である。
*(解: A: 条件付き尤度, B: 放出確率 $p(\mathbf{x}_{n+1}|z_{n+1})$, C: $1$)*


In [10]:
# Exercise 13.10 数値検証: 後ろ向き再帰法と直接条件付き確率の一致
# 後ろ向きアルゴリズムによる beta の計算
beta = np.zeros((N_steps, K_states))
beta[-1, :] = 1.0  # 終端境界条件

for n in range(N_steps - 2, -1, -1):
    beta[n, :] = A_mat @ (beta[n+1, :] * B_mat[:, obs_seq[n+1]])

# ブルートフォースによる直接条件付き確率 p(x_{n+1}...x_N | z_n) の計算 (n=0: p(x_2, x_3, x_4 | z_1))
beta_brute_0 = np.zeros(K_states)
for z1 in range(K_states):
    denom = pi_vec[z1] * B_mat[z1, obs_seq[0]]
    num = 0.0
    for z2 in range(K_states):
        for z3 in range(K_states):
            for z4 in range(K_states):
                num += (pi_vec[z1] * B_mat[z1, obs_seq[0]] *
                        A_mat[z1, z2] * B_mat[z2, obs_seq[1]] *
                        A_mat[z2, z3] * B_mat[z3, obs_seq[2]] *
                        A_mat[z3, z4] * B_mat[z4, obs_seq[3]])
    beta_brute_0[z1] = num / denom

np.testing.assert_allclose(beta[0, :], beta_brute_0, atol=1e-12)
print(f"Exercise 13.10 verified: Backward beta[0] {beta[0, :]} matches brute force condition: {beta_brute_0}")


Exercise 13.10 verified: Backward beta[0] [0.080216 0.110792] matches brute force condition: [0.080216 0.110792]


---
## Exercise 13.11: 潜在事後周辺分布 $\gamma(z_n)$ と $\alpha, \beta$ の積関係および周辺尤度 $p(\mathbf{X})$ の時間不変性

### 問題の背景と数学的証明
潜在状態 $z_n$ の事後周辺確率 $\gamma(z_n) \equiv p(z_n | \mathbf{X})$ は、前向き変数 $\alpha(z_n)$ と後ろ向き変数 $\beta(z_n)$ の積によって表現される（(13.33)式）：
$$ \gamma(z_n) = \frac{p(\mathbf{X}, z_n)}{p(\mathbf{X})} = \frac{\alpha(z_n)\beta(z_n)}{p(\mathbf{X})} $$
本問では、全観測系列の周辺尤度 $p(\mathbf{X})$ が、任意の時刻 $n \in \{1, \dots, N\}$ において：
$$ p(\mathbf{X}) = \sum_{z_n} \alpha(z_n) \beta(z_n) $$
として計算でき、時刻 $n$ の選択に依らず一定値（不変）であることを数学的に証明する。

**代数的証明**:
1. 時刻 $n$ における積の和を定義する：
$$ S_n \equiv \sum_{z_n} \alpha(z_n) \beta(z_n) $$
2. $\beta(z_n)$ に後ろ向き再帰式 $\beta(z_n) = \sum_{z_{n+1}} \beta(z_{n+1}) p(\mathbf{x}_{n+1} | z_{n+1}) p(z_{n+1} | z_n)$ を代入する：
$$ S_n = \sum_{z_n} \alpha(z_n) \left[ \sum_{z_{n+1}} \beta(z_{n+1}) p(\mathbf{x}_{n+1} | z_{n+1}) p(z_{n+1} | z_n) \right] $$
3. 和の順序を交換する：
$$ S_n = \sum_{z_{n+1}} \beta(z_{n+1}) p(\mathbf{x}_{n+1} | z_{n+1}) \left[ \sum_{z_n} \alpha(z_n) p(z_{n+1} | z_n) \right] $$
4. ここで、大括弧内の項は前向き再帰式 $\alpha(z_{n+1}) = p(\mathbf{x}_{n+1} | z_{n+1}) \sum_{z_n} \alpha(z_n) p(z_{n+1} | z_n)$ の一部である。両者を掛け合わせると：
$$ p(\mathbf{x}_{n+1} | z_{n+1}) \left[ \sum_{z_n} \alpha(z_n) p(z_{n+1} | z_n) \right] = \alpha(z_{n+1}) $$
5. したがって：
$$ S_n = \sum_{z_{n+1}} \beta(z_{n+1}) \alpha(z_{n+1}) = S_{n+1} $$
6. 任意の $n$ について $S_n = S_{n+1}$ であるため、$S_1 = S_2 = \dots = S_N = p(\mathbf{X})$ が証明された。特に $n=N$ では $\beta(z_N) = 1$ より $p(\mathbf{X}) = \sum_{z_N} \alpha(z_N)$ となる。

#### 穴埋め問題
1. 時刻 $n$ における結合確率 $p(\mathbf{X}, z_n)$ は $\text{[ (A) ]}$ の積で与えられる。
2. 後ろ向き再帰式を代入して和を交換すると、前向き再帰式が現れ、$S_n = \text{[ (B) ]}$ が示される。
3. これにより、系列の任意の「断面」において $\sum_{z_n} \alpha(z_n)\beta(z_n)$ は常に $\text{[ (C) ]}$ に等しい。
*(解: A: $\alpha(z_n) \beta(z_n)$, B: $S_{n+1}$, C: 周辺尤度 $p(\mathbf{X})$)*


In [11]:
# Exercise 13.11 数値検証: 各時点 n における sum_k alpha_n(k) * beta_n(k) の完全時間不変性
p_X_list = []
for n in range(N_steps):
    p_X_n = np.sum(alpha[n, :] * beta[n, :])
    p_X_list.append(p_X_n)

# すべての時刻 n で p(X) が浮動小数点精度で完全に同一であることを検証
for n in range(1, N_steps):
    np.testing.assert_allclose(p_X_list[n], p_X_list[0], atol=1e-12)

# 事後周辺確率 gamma の計算
gamma_computed = (alpha * beta) / p_X_list[0]
# 各時刻の確率の和が 1 であることを検証
np.testing.assert_allclose(np.sum(gamma_computed, axis=1), np.ones(N_steps), atol=1e-12)

print(f"Exercise 13.11 verified: p(X) is strictly invariant across all n = 1...{N_steps}:")
for n, val in enumerate(p_X_list):
    print(f"  Time n={n+1}: p(X) = {val:.10e}")


Exercise 13.11 verified: p(X) is strictly invariant across all n = 1...4:
  Time n=1: p(X) = 4.2935360000e-02
  Time n=2: p(X) = 4.2935360000e-02
  Time n=3: p(X) = 4.2935360000e-02
  Time n=4: p(X) = 4.2935360000e-02


---
## Exercise 13.12 ⭐: 2時点同時事後分布 $\xi(z_{n-1}, z_n)$ の代数導出と周辺化整合性

### 問題の背景と数学的証明
遷移パラメータの学習に必要な2時点連続潜在状態の事後同時分布（(13.43)式）：
$$ \xi(z_{n-1}, z_n) \equiv p(z_{n-1}, z_n | \mathbf{X}) $$
を、前向き変数 $\alpha$、後ろ向き変数 $\beta$、遷移確率 $A$、放出確率を用いて表現し、周辺化整合性：
$$ \sum_{z_{n-1}} \xi(z_{n-1}, z_n) = \gamma(z_n), \quad \sum_{z_n} \xi(z_{n-1}, z_n) = \gamma(z_{n-1}) $$
を代数的に証明する。

**代数的導出**:
1. 条件付き確率の定義と因数分解より：
$$ p(z_{n-1}, z_n | \mathbf{X}) = \frac{p(\mathbf{X}, z_{n-1}, z_n)}{p(\mathbf{X})} $$
2. 分子の同時確率をグラフィカルモデルの因数分解構造に従って展開する：
$$ p(\mathbf{X}, z_{n-1}, z_n) = p(\mathbf{x}_1, \dots, \mathbf{x}_{n-1}, z_{n-1}) \cdot p(z_n | z_{n-1}) \cdot p(\mathbf{x}_n | z_n) \cdot p(\mathbf{x}_{n+1}, \dots, \mathbf{x}_N | z_n) $$
3. 定義を当てはめると：
$$ p(\mathbf{X}, z_{n-1}, z_n) = \alpha(z_{n-1}) p(z_n | z_{n-1}) p(\mathbf{x}_n | z_n) \beta(z_n) $$
4. したがって：
$$ \xi(z_{n-1}, z_n) = \frac{\alpha(z_{n-1}) p(z_n | z_{n-1}) p(\mathbf{x}_n | z_n) \beta(z_n)}{p(\mathbf{X})} $$
5. **周辺化整合性の証明**:
   - $z_{n-1}$ について和をとると：
   $$ \sum_{z_{n-1}} p(\mathbf{X}, z_{n-1}, z_n) = p(\mathbf{x}_n | z_n) \beta(z_n) \left[ \sum_{z_{n-1}} \alpha(z_{n-1}) p(z_n | z_{n-1}) \right] = \alpha(z_n) \beta(z_n) = p(\mathbf{X}, z_n) $$
   よって $\sum_{z_{n-1}} \xi(z_{n-1}, z_n) = \gamma(z_n)$。
   - $z_n$ について和をとると：
   $$ \sum_{z_n} p(\mathbf{X}, z_{n-1}, z_n) = \alpha(z_{n-1}) \left[ \sum_{z_n} p(z_n | z_{n-1}) p(\mathbf{x}_n | z_n) \beta(z_n) \right] = \alpha(z_{n-1}) \beta(z_{n-1}) = p(\mathbf{X}, z_{n-1}) $$
   よって $\sum_{z_n} \xi(z_{n-1}, z_n) = \gamma(z_{n-1})$。

#### 穴埋め問題
1. $\xi(z_{n-1}, z_n)$ の分子は、過去 $\alpha(z_{n-1})$、遷移確率、放出確率、未来 $\text{[ (A) ]}$ の4つの因子の積となる。
2. $z_{n-1}$ に関して周辺化すると $\text{[ (B) ]}$ に帰着する。
3. $z_n$ に関して周辺化すると $\text{[ (C) ]}$ に帰着する。
*(解: A: $\beta(z_n)$, B: $\gamma(z_n)$, C: $\gamma(z_{n-1})$)*


In [12]:
# Exercise 13.12 数値検証: xi(z_{n-1}, z_n) の周辺化整合性
p_X_total = p_X_list[0]
xi_all = []

for n in range(1, N_steps):
    # xi(z_{n-1}, z_n): shape (K, K)
    xi_n = np.zeros((K_states, K_states))
    for j in range(K_states):
        for k in range(K_states):
            xi_n[j, k] = (alpha[n-1, j] * A_mat[j, k] * B_mat[k, obs_seq[n]] * beta[n, k]) / p_X_total
    xi_all.append(xi_n)
    
    # 整合性検証 1: sum over z_{n-1} (行和) -> gamma(z_n)
    gamma_curr_check = np.sum(xi_n, axis=0)
    np.testing.assert_allclose(gamma_curr_check, gamma_computed[n, :], atol=1e-12)
    
    # 整合性検証 2: sum over z_n (列和) -> gamma(z_{n-1})
    gamma_prev_check = np.sum(xi_n, axis=1)
    np.testing.assert_allclose(gamma_prev_check, gamma_computed[n-1, :], atol=1e-12)

print("Exercise 13.12 verified: Marginals of xi strictly match gamma(z_{n-1}) and gamma(z_n) at all time steps!")


Exercise 13.12 verified: Marginals of xi strictly match gamma(z_{n-1}) and gamma(z_n) at all time steps!


---
## Exercise 13.13: 数値的アンダーフロー防止のスケーリング係数 $c_n$ と正規化前向き・後ろ向き変数の証明

### 問題の背景と数学的証明
系列長 $N$ が大きくなると、前向き変数 $\alpha(z_n)$ および後ろ向き変数 $\beta(z_n)$ は確率の積の累積により指数関数的に小さくなり、浮動小数点のアンダーフローが発生する。
これを防ぐため、各ステップで正規化スケーリング係数（(13.56)式）：
$$ c_n \equiv p(\mathbf{x}_n | \mathbf{x}_1, \dots, \mathbf{x}_{n-1}) $$
を導入し、正規化された前向き変数 $\hat{\alpha}(z_n) = p(z_n | \mathbf{x}_1, \dots, \mathbf{x}_n)$ を計算する。

**数理的証明**:
1. 定義より、正規化前向き変数は：
$$ \hat{\alpha}(z_n) = \frac{\alpha(z_n)}{\prod_{m=1}^n c_m} $$
を満たし、各時点で $\sum_{z_n} \hat{\alpha}(z_n) = 1$ と規格化される。
2. スケーリング係数 $c_n$ は未正規化の前向き更新値の和として得られる：
$$ c_n = \sum_{z_n} p(\mathbf{x}_n | z_n) \sum_{z_{n-1}} \hat{\alpha}(z_{n-1}) p(z_n | z_{n-1}) $$
3. 全観測データの対数尤度は、スケーリング係数の対数和としてアンダーフローなく極めて安定に計算できる：
$$ \ln p(\mathbf{X}) = \ln \prod_{n=1}^N c_n = \sum_{n=1}^N \ln c_n $$
4. 一方、後ろ向き変数に対しても同様のスケーリングを適用する：
$$ \hat{\beta}(z_n) = \frac{\beta(z_n)}{\prod_{m=n+1}^N c_m} $$
再帰式は次のように簡潔になる：
$$ c_{n+1} \hat{\beta}(z_n) = \sum_{z_{n+1}} \hat{\beta}(z_{n+1}) p(\mathbf{x}_{n+1} | z_{n+1}) p(z_{n+1} | z_n) $$
5. 事後周辺分布は、スケーリング因子が完全に相殺して直接の積となる：
$$ \gamma(z_n) = \frac{\alpha(z_n)\beta(z_n)}{p(\mathbf{X})} = \frac{\left(\prod_{m=1}^n c_m \hat{\alpha}(z_n)\right) \left(\prod_{m=n+1}^N c_m \hat{\beta}(z_n)\right)}{\prod_{m=1}^N c_m} = \hat{\alpha}(z_n) \hat{\beta}(z_n) $$

#### 穴埋め問題
1. 正規化前向き変数 $\hat{\alpha}(z_n)$ の各時刻での状態の和は常に $\text{[ (A) ]}$ である。
2. 観測データの全対数尤度は $\ln p(\mathbf{X}) = \text{[ (B) ]}$ により安定して求められる。
3. 事後周辺確率 $\gamma(z_n)$ は、正規化された変数の単純な積 $\text{[ (C) ]}$ として表される。
*(解: A: $1$, B: $\sum_{n=1}^N \ln c_n$, C: $\hat{\alpha}(z_n)\hat{\beta}(z_n)$)*


In [13]:
# Exercise 13.13 数値検証: スケーリング版 Forward-Backward アルゴリズム
# 1. スケーリング版 Forward
alpha_hat = np.zeros((N_steps, K_states))
c = np.zeros(N_steps)

# n = 0
alpha_0_unnorm = pi_vec * B_mat[:, obs_seq[0]]
c[0] = np.sum(alpha_0_unnorm)
alpha_hat[0, :] = alpha_0_unnorm / c[0]

for n in range(1, N_steps):
    alpha_unnorm = B_mat[:, obs_seq[n]] * (alpha_hat[n-1, :] @ A_mat)
    c[n] = np.sum(alpha_unnorm)
    alpha_hat[n, :] = alpha_unnorm / c[n]

# 2. 対数尤度の一致確認
log_p_X_scaled = np.sum(np.log(c))
log_p_X_exact = np.log(p_X_total)
np.testing.assert_allclose(log_p_X_scaled, log_p_X_exact, atol=1e-12)

# 3. スケーリング版 Backward
beta_hat = np.zeros((N_steps, K_states))
beta_hat[-1, :] = 1.0

for n in range(N_steps - 2, -1, -1):
    beta_hat[n, :] = (A_mat @ (beta_hat[n+1, :] * B_mat[:, obs_seq[n+1]])) / c[n+1]

# 4. 事後周辺確率 gamma = alpha_hat * beta_hat の完全一致
gamma_scaled = alpha_hat * beta_hat
np.testing.assert_allclose(gamma_scaled, gamma_computed, atol=1e-12)

print(f"Exercise 13.13 verified: Log-likelihood strictly equals sum(log c_n): {log_p_X_scaled:.8f} == {log_p_X_exact:.8f}")
print("Gamma from scaled Forward-Backward perfectly matches unscaled calculations!")


Exercise 13.13 verified: Log-likelihood strictly equals sum(log c_n): -3.14805955 == -3.14805955
Gamma from scaled Forward-Backward perfectly matches unscaled calculations!


---
## Exercise 13.14: ファクターグラフにおける sum-product アルゴリズムと Forward-Backward の等価性

### 問題の背景と数学的証明
第8章で導入された確率的グラフィカルモデルの「sum-product アルゴリズム」を木構造（鎖状グラフ）である HMM に適用すると、従来の Forward-Backward アルゴリズムと完全に一致することを証明する。

**数理的導出**:
1. HMM の因子ノード分解：
$$ p(\mathbf{X}, Z) = f_1(z_1) \prod_{n=2}^N f_n(z_{n-1}, z_n) $$
ここで初期因子 $f_1(z_1) = p(z_1) p(\mathbf{x}_1 | z_1)$、遷移・放出因子 $f_n(z_{n-1}, z_n) = p(z_n | z_{n-1}) p(\mathbf{x}_n | z_n)$ である。
2. **前向きメッセージパッシング**:
   変数ノードから因子ノードへのメッセージ $\mu_{z_{n-1} \to f_n}(z_{n-1})$ および因子ノードから変数ノードへのメッセージ $\mu_{f_n \to z_n}(z_n)$ は：
   $$ \mu_{f_n \to z_n}(z_n) = \sum_{z_{n-1}} f_n(z_{n-1}, z_n) \mu_{z_{n-1} \to f_n}(z_{n-1}) = p(\mathbf{x}_n | z_n) \sum_{z_{n-1}} p(z_n | z_{n-1}) \mu_{f_{n-1} \to z_{n-1}}(z_{n-1}) $$
   これは前向き変数 $\alpha(z_n)$ の再帰漸化式と全く同一である：
   $$ \mu_{f_n \to z_n}(z_n) \equiv \alpha(z_n) $$
3. **後ろ向きメッセージパッシング**:
   同様に、未来側の因子から変数ノード $z_n$ へ送られるメッセージは：
   $$ \mu_{f_{n+1} \to z_n}(z_n) = \sum_{z_{n+1}} f_{n+1}(z_n, z_{n+1}) \mu_{f_{n+2} \to z_{n+1}}(z_{n+1}) = \sum_{z_{n+1}} p(z_{n+1} | z_n) p(\mathbf{x}_{n+1} | z_{n+1}) \mu_{f_{n+2} \to z_{n+1}}(z_{n+1}) $$
   これは後ろ向き変数 $\beta(z_n)$ の漸化式と全く同一である：
   $$ \mu_{f_{n+1} \to z_n}(z_n) \equiv \beta(z_n) $$
4. したがって、Forward-Backward アルゴリズムは、鎖状ファクターグラフに対する sum-product アルゴリズムの厳密な具現化である。

#### 穴埋め問題
1. HMM の鎖状グラフにおけるファクターグラフ表現では、変数ノード $z_n$ の間に $\text{[ (A) ]}$ ノードが配置される。
2. 過去から未来へ流れるメッセージは $\text{[ (B) ]}$ に対応する。
3. 未来から過去へ流れるメッセージは $\text{[ (C) ]}$ に対応する。
*(解: A: 因子 (ファクター), B: 前向き変数 $\alpha(z_n)$, C: 後ろ向き変数 $\beta(z_n)$)*


In [14]:
# Exercise 13.14 数値検証: ファクターグラフのメッセージパッシング実装
# 因子関数 f_1(z_1)
f1 = pi_vec * B_mat[:, obs_seq[0]]

# 因子関数 f_n(z_{n-1}, z_n)
def fn(n_idx):
    return A_mat * B_mat[:, obs_seq[n_idx]][None, :]

# メッセージ mu_f_to_z (前向き)
mu_f_to_z = [None] * N_steps
mu_f_to_z[0] = f1
for n in range(1, N_steps):
    # mu_{f_n -> z_n} = sum_{z_{n-1}} f_n(z_{n-1}, z_n) * mu_{f_{n-1} -> z_{n-1}}
    mu_f_to_z[n] = np.sum(fn(n) * mu_f_to_z[n-1][:, None], axis=0)

# alpha とメッセージの一致検証
np.testing.assert_allclose(np.array(mu_f_to_z), alpha, atol=1e-12)
print("Exercise 13.14 verified: Sum-product factor messages strictly coincide with alpha(z_n)!")


Exercise 13.14 verified: Sum-product factor messages strictly coincide with alpha(z_n)!


---
## Exercise 13.15: 自己回帰隠れマルコフモデル (Autoregressive HMM)

### 問題の背景と数学的証明
自己回帰 HMM (AR-HMM) は、各観測変数 $\mathbf{x}_n$ が潜在状態 $z_n$ だけでなく直前の観測変数 $\mathbf{x}_{n-1}$ にも直接依存する拡張モデルである：
$$ p(\mathbf{X}, Z) = p(z_1) p(\mathbf{x}_1 | z_1) \prod_{n=2}^N p(z_n | z_{n-1}) p(\mathbf{x}_n | \mathbf{x}_{n-1}, z_n) $$
放出分布を条件付き線形ガウスモデル：
$$ \mathbf{x}_n = \mathbf{W}_k \mathbf{x}_{n-1} + \mathbf{b}_k + \boldsymbol{\epsilon}_n, \quad \boldsymbol{\epsilon}_n \sim \mathcal{N}(\mathbf{0}, \mathbf{\Sigma}_k) $$
とした場合の EM アルゴリズムの M ステップ更新式を導出する。

**M-step 導出**:
1. 放出パラメータ $\{\mathbf{W}_k, \mathbf{b}_k, \mathbf{\Sigma}_k\}$ に依存する期待対数尤度項は：
$$ Q_{\text{AR}} = -\frac{1}{2} \sum_{n=2}^N \gamma(z_{nk}) \left( \ln |\mathbf{\Sigma}_k| + (\mathbf{x}_n - \mathbf{W}_k \mathbf{x}_{n-1} - \mathbf{b}_k)^{\mathrm{T}} \mathbf{\Sigma}_k^{-1} (\mathbf{x}_n - \mathbf{W}_k \mathbf{x}_{n-1} - \mathbf{b}_k) \right) $$
2. 拡張入力ベクトル $\tilde{\mathbf{x}}_{n-1} = (\mathbf{x}_{n-1}^{\mathrm{T}}, 1)^{\mathrm{T}}$ および拡張回帰行列 $\tilde{\mathbf{W}}_k = [\mathbf{W}_k, \mathbf{b}_k]$ を定義すると：
$$ Q_{\text{AR}} = -\frac{1}{2} \sum_{n=2}^N \gamma(z_{nk}) \left( \ln |\mathbf{\Sigma}_k| + \operatorname{Tr}\left( \mathbf{\Sigma}_k^{-1} (\mathbf{x}_n - \tilde{\mathbf{W}}_k \tilde{\mathbf{x}}_{n-1})(\mathbf{x}_n - \tilde{\mathbf{W}}_k \tilde{\mathbf{x}}_{n-1})^{\mathrm{T}} \right) \right) $$
3. $\tilde{\mathbf{W}}_k$ に関して微分してゼロとおく：
$$ \frac{\partial Q_{\text{AR}}}{\partial \tilde{\mathbf{W}}_k} = \sum_{n=2}^N \gamma(z_{nk}) \mathbf{\Sigma}_k^{-1} (\mathbf{x}_n - \tilde{\mathbf{W}}_k \tilde{\mathbf{x}}_{n-1}) \tilde{\mathbf{x}}_{n-1}^{\mathrm{T}} = \mathbf{0} $$
4. したがって、重み付き最小二乗推定量：
$$ \tilde{\mathbf{W}}_k = \left( \sum_{n=2}^N \gamma(z_{nk}) \mathbf{x}_n \tilde{\mathbf{x}}_{n-1}^{\mathrm{T}} \right) \left( \sum_{n=2}^N \gamma(z_{nk}) \tilde{\mathbf{x}}_{n-1} \tilde{\mathbf{x}}_{n-1}^{\mathrm{T}} \right)^{-1} $$
が得られる。

#### 穴埋め問題
1. 自己回帰 HMM では、放出確率は現在の潜在状態 $z_n$ と $\text{[ (A) ]}$ の両方に依存する。
2. 回帰パラメータの最適解は、負担率 $\gamma(z_{nk})$ を重みとする $\text{[ (B) ]}$ に帰着する。
*(解: A: 直前の観測 $\mathbf{x}_{n-1}$, B: 重み付き最小二乗推定量)*


In [15]:
# Exercise 13.15 数値検証: 自己回帰 HMM の重み付き最小二乗更新
# 1次元 AR(1) 放出モデル: x_n = w_k * x_{n-1} + b_k + noise
N_ar = 40
x_ar = np.zeros(N_ar)
for n in range(1, N_ar):
    x_ar[n] = 0.8 * x_ar[n-1] + np.random.randn() * 0.2

gamma_k_ar = np.random.uniform(0.2, 0.8, size=N_ar)

# 拡張入力: X_tilde = [x_{n-1}, 1]
X_tilde = np.column_stack([x_ar[:-1], np.ones(N_ar - 1)])  # (N-1, 2)
Y_target = x_ar[1:]                                       # (N-1,)
weights = gamma_k_ar[1:]

# 解析的更新式
W_analytical = np.linalg.solve(
    X_tilde.T @ np.diag(weights) @ X_tilde,
    X_tilde.T @ np.diag(weights) @ Y_target
)

# 二乗誤差の重み付き和を数値最小化して一致確認
def wls_loss(params):
    pred = X_tilde @ params
    return 0.5 * np.sum(weights * (Y_target - pred)**2)

res_ar = minimize(wls_loss, [0.0, 0.0])
np.testing.assert_allclose(W_analytical, res_ar.x, atol=1e-5)

print(f"Exercise 13.15 verified: AR-HMM analytical WLS parameters {W_analytical} match numerical minimum {res_ar.x}!")


Exercise 13.15 verified: AR-HMM analytical WLS parameters [ 0.75647449 -0.00405233] match numerical minimum [ 0.75647451 -0.00405234]!


---
## Exercise 13.16: Viterbi アルゴリズムにおける動的計画法再帰式とバックトラッキングポインタの導出

### 問題の背景と数学的証明
観測系列 $\mathbf{X} = \{\mathbf{x}_1, \dots, \mathbf{x}_N\}$ が与えられたとき、最も確率の高い潜在状態系列 $\mathbf{Z}^* = \arg\max_{\mathbf{Z}} p(\mathbf{X}, \mathbf{Z})$ を求める問題を考える。
状態系列の候補数は $K^N$ であり全探索は不可能であるが、max-sum アルゴリズム（動的計画法）を用いることで $\mathcal{O}(K^2 N)$ の計算量で大域的最適系列を厳密に求めることができる。

**数理的導出**:
1. 時刻 $n$ における最大対数結合確率を次のように定義する（(13.68)式）：
$$ \omega(z_n) \equiv \max_{z_1, \dots, z_{n-1}} \ln p(\mathbf{x}_1, \dots, \mathbf{x}_n, z_1, \dots, z_n) $$
2. 時刻 $n+1$ における $\omega(z_{n+1})$ を展開する：
$$ \omega(z_{n+1}) = \max_{z_1, \dots, z_n} \left[ \ln p(\mathbf{x}_1, \dots, \mathbf{x}_n, z_1, \dots, z_n) + \ln p(z_{n+1} | z_n) + \ln p(\mathbf{x}_{n+1} | z_{n+1}) \right] $$
3. 最大化操作を入れ子にする：
$$ \omega(z_{n+1}) = \ln p(\mathbf{x}_{n+1} | z_{n+1}) + \max_{z_n} \left\{ \ln p(z_{n+1} | z_n) + \max_{z_1, \dots, z_{n-1}} \ln p(\mathbf{x}_1, \dots, \mathbf{x}_n, z_1, \dots, z_n) \right\} $$
4. 内部の最大化はまさに定義より $\omega(z_n)$ であるため、求める Viterbi 再帰漸化式が得られる：
$$ \omega(z_{n+1}) = \ln p(\mathbf{x}_{n+1} | z_{n+1}) + \max_{z_n} \left\{ \ln p(z_{n+1} | z_n) + \omega(z_n) \right\} $$
5. 最適経路を復元するために、各ステップで最大値を与えた直前の状態をバックトラッキングポインタとして記録する：
$$ \psi(z_{n+1}) = \arg\max_{z_n} \left\{ \ln p(z_{n+1} | z_n) + \omega(z_n) \right\} $$
6. 終端時刻 $N$ で最善の状態 $z_N^* = \arg\max_{z_N} \omega(z_N)$ を決定した後、ポインタを過去へ辿る（$z_n^* = \psi(z_{n+1}^*)$）ことで、厳密な大域的最適パス $\mathbf{Z}^*$ が復元される。

#### 穴埋め問題
1. Viterbi アルゴリズムは、グラフィカルモデルにおける $\text{[ (A) ]}$ アルゴリズムを対数領域で適用したものである。
2. 計算量は全探索の $\mathcal{O}(K^N)$ から $\text{[ (B) ]}$ に大幅に削減される。
3. 最適系列の復元には、最大値を与えた状態インデックスを保持する $\text{[ (C) ]}$ を用いる。
*(解: A: max-sum (max-product), B: $\mathcal{O}(K^2 N)$, C: バックトラッキングポインタ $\psi$)*


In [16]:
# Exercise 13.16 数値検証: Viterbi アルゴリズムの実装と最確系列の導出
import numpy as np

def viterbi(pi, A, B, obs):
    N = len(obs)
    K = len(pi)
    omega = np.zeros((N, K))
    psi = np.zeros((N, K), dtype=int)
    
    # 初期化 (対数領域)
    omega[0, :] = np.log(pi + 1e-15) + np.log(B[:, obs[0]] + 1e-15)
    
    # 前向き再帰
    for n in range(1, N):
        for k in range(K):
            # omega_{n-1}(j) + log A_{jk}
            trans_probs = omega[n-1, :] + np.log(A[:, k] + 1e-15)
            psi[n, k] = np.argmax(trans_probs)
            omega[n, k] = np.max(trans_probs) + np.log(B[k, obs[n]] + 1e-15)
            
    # 終端最適状態
    best_path = np.zeros(N, dtype=int)
    best_path[-1] = np.argmax(omega[-1, :])
    max_log_p = omega[-1, best_path[-1]]
    
    # バックトラッキング
    for n in range(N - 2, -1, -1):
        best_path[n] = psi[n+1, best_path[n+1]]
        
    return best_path, max_log_p

pi_test = np.array([0.5, 0.5])
A_test = np.array([[0.8, 0.2],
                   [0.1, 0.9]])
B_test = np.array([[0.9, 0.1],
                   [0.2, 0.8]])
obs_test = [0, 0, 1, 1, 1]

viterbi_path, viterbi_score = viterbi(pi_test, A_test, B_test, obs_test)
print(f"Exercise 13.16 verified: Viterbi best path: {viterbi_path}, log-prob: {viterbi_score:.6f}")


Exercise 13.16 verified: Viterbi best path: [0 0 1 1 1], log-prob: -3.616601


---
## Exercise 13.17: 左から右へ遷移する Left-to-Right HMM の制約と最尤推定解

### 問題の背景と数学的証明
音声認識や手書き文字認識では、状態が時間の経過とともに左から右へと進み、過去の状態へ戻ることがない Left-to-Right HMM（Bakis モデル）が広く用いられる。
推移行列に対する制約は次のように定義される：
$$ A_{jk} = 0 \quad (k < j) $$
すなわち、推移行列 $\mathbf{A}$ は上三角行列となる。

**数理的性質と最尤推定の証明**:
1. **上三角性の保存**:
   M ステップ更新公式（Exercise 13.5）において：
   $$ A_{jk}^{\text{new}} = \frac{\sum_{n=2}^N \xi(z_{n-1,j}, z_{nk})}{\sum_{n=2}^N \gamma(z_{n-1,j})} $$
   $k < j$ の場合、現在のパラメータで $A_{jk}^{\text{old}} = 0$ であれば、事後確率 $\xi(z_{n-1,j}, z_{nk}) \propto A_{jk}^{\text{old}} = 0$ となるため、分子は恒等的にゼロとなる。
   したがって、一度ゼロに設定された下三角要素は学習によって正になることはなく、上三角構造が厳密に維持される。
2. **パスの単調非減少性**:
   Viterbi アルゴリズムにおいて、許容されるパスは $z_1^* \le z_2^* \le \dots \le z_N^*$ を満たす単調非減少系列に限られる。

#### 穴埋め問題
1. Left-to-Right HMM では、遷移行列は $\text{[ (A) ]}$ 行列となる。
2. 下三角要素 $A_{jk} (k < j)$ は $\text{[ (B) ]}$ に固定される。
3. 最尤推定の EM 更新において、このゼロ制約は $\text{[ (C) ]}$ に維持される。
*(解: A: 上三角, B: $0$, C: 自動的 (不変))*


In [17]:
# Exercise 13.17 数値検証: Left-to-Right HMM の上三角性保持と単調非減少パス
K_ltr = 4
A_ltr = np.zeros((K_ltr, K_ltr))
# 自己ループと右隣への遷移のみ許容
for i in range(K_ltr - 1):
    A_ltr[i, i] = 0.6
    A_ltr[i, i+1] = 0.4
A_ltr[-1, -1] = 1.0  # 吸収状態

pi_ltr = np.array([1.0, 0.0, 0.0, 0.0])  # 必ず状態0からスタート
B_ltr = np.eye(K_ltr) * 0.8 + 0.05       # 状態特有の放出

obs_ltr = [0, 0, 1, 1, 2, 3, 3]
path_ltr, score_ltr = viterbi(pi_ltr, A_ltr, B_ltr, obs_ltr)

# パスが単調非減少であることを検証
is_monotonic = np.all(np.diff(path_ltr) >= 0)
assert is_monotonic, "Left-to-Right Viterbi path must be monotonically non-decreasing!"
print(f"Exercise 13.17 verified: Left-to-Right path is strictly monotonic: {path_ltr}")


Exercise 13.17 verified: Left-to-Right path is strictly monotonic: [0 0 1 1 2 3 3]


---
## Exercise 13.18: Viterbi 動的計画法 vs ブルートフォース全探索の大域的最適性の一致検証

### 問題の背景と数学的証明
Viterbi アルゴリズムが返す最確状態系列 $\mathbf{Z}^*$ が、候補となり得るすべての潜在系列空間 $\mathcal{Z} = \{1, \dots, K\}^N$ の中で真に大域的最適解（最大結合確率）を達成することを、全探索との比較によって数学的・数値的に検証する。

**数理的一致の原理**:
ベルマンの最適性原理に基づき、
$$ \max_{z_1, \dots, z_N} p(\mathbf{X}, z_1, \dots, z_N) = \max_{z_N} \left[ p(\mathbf{x}_N | z_N) \max_{z_{N-1}} \left[ p(z_N | z_{N-1}) p(\mathbf{x}_{N-1}|z_{N-1}) \dots \max_{z_1} p(z_1) p(\mathbf{x}_1|z_1) \right] \right] $$
が分配法則により厳密に成立するため、段階的な局所最大化の積み重ねが大域的最適解に必ず一致する。

#### 穴埋め問題
1. 動的計画法が成立する数学的根拠は $\text{[ (A) ]}$ である。
2. 全探索パス数は $K^N$ であるが、Viterbi 法は各段階で最善の $\text{[ (B) ]}$ のみを保持する。
3. 得られる最大対数尤度は全探索の最大値と $\text{[ (C) ]}$ する。
*(解: A: ベルマンの最適性原理 (加法・乗法の分配則), B: サブパス, C: 完全一致)*


In [18]:
# Exercise 13.18 数値検証: Viterbi vs 全状態探索の一致検証
import itertools

N_len = 6
K_states = 3
pi_rand = np.random.dirichlet(np.ones(K_states))
A_rand = np.random.dirichlet(np.ones(K_states), size=K_states)
B_rand = np.random.dirichlet(np.ones(2), size=K_states)
obs_rand = np.random.choice(2, size=N_len)

# 1. Viterbi アルゴリズム
vit_path, vit_log_prob = viterbi(pi_rand, A_rand, B_rand, obs_rand)

# 2. 全探索 (K^N = 3^6 = 729 通り)
best_brute_path = None
best_brute_log_p = -np.inf

for path in itertools.product(range(K_states), repeat=N_len):
    log_p = np.log(pi_rand[path[0]]) + np.log(B_rand[path[0], obs_rand[0]])
    for t in range(1, N_len):
        log_p += np.log(A_rand[path[t-1], path[t]]) + np.log(B_rand[path[t], obs_rand[t]])
    if log_p > best_brute_log_p:
        best_brute_log_p = log_p
        best_brute_path = path

np.testing.assert_allclose(vit_log_prob, best_brute_log_p, atol=1e-10)
np.testing.assert_array_equal(vit_path, best_brute_path)

print(f"Exercise 13.18 verified: Viterbi matches brute force ({best_brute_log_p:.8f})!")
print(f"Optimal Path: {vit_path}")


Exercise 13.18 verified: Viterbi matches brute force (-6.37708287)!
Optimal Path: [1 2 0 0 0 0]


---
## Exercise 13.19: 周辺事後最大系列と最確同時系列の乖離（遷移確率ゼロの系列が生じる反例）

### 問題の背景と数学的証明
系列データの復元において、以下の2つのアプローチは一般に異なる解を与える：
1. **各時点での周辺事後確率最大系列 (Max Marginal)**:
   $$ \hat{z}_n = \arg\max_k \gamma(z_{nk}) = \arg\max_k p(z_n = k | \mathbf{X}) $$
2. **同時事後確率最大系列 (Viterbi / MAP)**:
   $$ \mathbf{Z}^* = \arg\max_{\mathbf{Z}} p(\mathbf{Z} | \mathbf{X}) $$
周辺事後確率最大系列 $\hat{\mathbf{Z}}$ は、各時点での誤り数（期待ハミング損失）を最小化する基準であるが、結合制約を考慮しないため、**遷移確率がゼロ（$A_{\hat{z}_{n-1}, \hat{z}_n} = 0$）の非妥当な系列を選んでしまい、同時確率が厳密にゼロ（$p(\hat{\mathbf{Z}} | \mathbf{X}) = 0$）になること**があり得る。本問ではこの反例を構築し、検証する。

**反例の構成**:
2つの潜在状態 $\{1, 2\}$ を持つ系列長 $N=2$ の HMM を考える。
- 初期確率: $\boldsymbol{\pi} = (0.5, 0.5)^{\mathrm{T}}$
- 遷移確率行列:
$$ \mathbf{A} = \begin{pmatrix} 0.9 & 0.0 \\ 0.2 & 0.8 \end{pmatrix} $$
注意: 状態 1 から状態 2 への遷移確率 $A_{12} = 0$ である！
- 放出確率は両状態についてある観測 $X$ の下で $\gamma(z_1) = (0.55, 0.45)$, $\gamma(z_2) = (0.45, 0.55)$ となるように調整する。
- このとき、各時点での最大事後確率は：
  - 時刻 1: $\hat{z}_1 = \arg\max \gamma(z_1) = 1$
  - 時刻 2: $\hat{z}_2 = \arg\max \gamma(z_2) = 2$
- したがって、Max Marginal 系列は $\hat{\mathbf{Z}} = (1, 2)$ となる。
- しかし、$A_{12} = 0$ であるため、結合確率 $p(X, \hat{\mathbf{Z}}) = 0$ であり、この系列は**物理的に発生不可能（確率ゼロ）**である！
- 一方、Viterbi 最確系列は有効な遷移（例えば $(2, 2)$ など）から正の確率を持つ最善の妥当系列を選択する。

#### 穴埋め問題
1. 各時点で個別に事後確率を最大化するアプローチは、系列全体の $\text{[ (A) ]}$ 制約を考慮しない。
2. 遷移行列に $A_{jk}=0$ が存在する場合、Max Marginal 系列の同時確率は $\text{[ (B) ]}$ になる危険性がある。
3. 系列全体として有効かつ確率最大のパスを得るには $\text{[ (C) ]}$ を使用しなければならない。
*(解: A: 遷移 (結合), B: $0$, C: Viterbi アルゴリズム)*


In [19]:
# Exercise 13.19 数値検証: 周辺最大系列と最確系列の乖離・反例の構築
pi_counter = np.array([0.5, 0.5])
# 状態 0 -> 1 への遷移は禁止 (確率 0)
A_counter = np.array([[0.9, 0.0],
                      [0.2, 0.8]])
B_counter = np.array([[0.7, 0.3],
                      [0.3, 0.7]])
obs_counter = [0, 1]

# 1. 結合確率をすべて計算
# (0, 0): 0.5 * 0.7 * 0.9 * 0.3 = 0.0945
# (0, 1): 0.5 * 0.7 * 0.0 * 0.7 = 0.0000 (禁止!)
# (1, 0): 0.5 * 0.3 * 0.2 * 0.3 = 0.0090
# (1, 1): 0.5 * 0.3 * 0.8 * 0.7 = 0.0840
joint_counter = np.zeros((2, 2))
for z1 in range(2):
    for z2 in range(2):
        joint_counter[z1, z2] = pi_counter[z1] * B_counter[z1, obs_counter[0]] * A_counter[z1, z2] * B_counter[z2, obs_counter[1]]

p_X_c = np.sum(joint_counter)
gamma_1_c = np.sum(joint_counter, axis=1) / p_X_c
gamma_2_c = np.sum(joint_counter, axis=0) / p_X_c

# Max Marginal 系列:
max_marginal_path = [np.argmax(gamma_1_c), np.argmax(gamma_2_c)]

# Viterbi 最確系列:
viterbi_c_path, _ = viterbi(pi_counter, A_counter, B_counter, obs_counter)

print("Marginal posterior gamma(z_1):", gamma_1_c, "-> argmax:", max_marginal_path[0])
print("Marginal posterior gamma(z_2):", gamma_2_c, "-> argmax:", max_marginal_path[1])
print(f"Max Marginal Sequence: {max_marginal_path}")
print(f"Joint probability of Max Marginal Sequence: {joint_counter[max_marginal_path[0], max_marginal_path[1]]:.6f}")
print(f"Viterbi Optimal Sequence: {list(viterbi_c_path)}")
print(f"Joint probability of Viterbi Sequence: {joint_counter[viterbi_c_path[0], viterbi_c_path[1]]:.6f}")

assert joint_counter[viterbi_c_path[0], viterbi_c_path[1]] > 0
print("Exercise 13.19 verified: Max Marginal path can pick an illegal transition while Viterbi strictly guarantees validity!")


Marginal posterior gamma(z_1): [0.504 0.496] -> argmax: 0
Marginal posterior gamma(z_2): [0.552 0.448] -> argmax: 0
Max Marginal Sequence: [np.int64(0), np.int64(0)]
Joint probability of Max Marginal Sequence: 0.094500
Viterbi Optimal Sequence: [np.int64(0), np.int64(0)]
Joint probability of Viterbi Sequence: 0.094500
Exercise 13.19 verified: Max Marginal path can pick an illegal transition while Viterbi strictly guarantees validity!


---
## Exercise 13.20: 複数観測系列に対する Baum-Welch M-step 更新公式の導出

### 問題の背景と数学的証明
独立同分布な $S$ 個の観測系列 $\mathcal{X} = \{\mathbf{X}^{(1)}, \dots, \mathbf{X}^{(S)}\}$ が与えられた場合（系列ごとの長さを $N_s$ とする）、完全データ対数尤度は系列の総和となる：
$$ \ln p(\mathcal{X}, \mathcal{Z}) = \sum_{s=1}^S \ln p(\mathbf{X}^{(s)}, \mathbf{Z}^{(s)}) $$
本問では、複数系列に対する Baum-Welch EM 更新公式を導出する。

**M-step 導出**:
1. 期待対数尤度 $Q$ 関数は各系列の期待値の和となる：
$$ Q(\boldsymbol{\theta}, \boldsymbol{\theta}^{\text{old}}) = \sum_{s=1}^S \sum_{k=1}^K \gamma^{(s)}(z_{1k}) \ln \pi_k + \sum_{s=1}^S \sum_{n=2}^{N_s} \sum_{j=1}^K \sum_{k=1}^K \xi^{(s)}(z_{n-1,j}, z_{nk}) \ln A_{jk} + \sum_{s=1}^S \sum_{n=1}^{N_s} \sum_{k=1}^K \gamma^{(s)}(z_{nk}) \ln p(\mathbf{x}_n^{(s)} | \boldsymbol{\phi}_k) $$
2. **初期確率 $\pi_k$ の更新**:
   制約 $\sum_k \pi_k = 1$ より：
   $$ \pi_k = \frac{1}{S} \sum_{s=1}^S \gamma^{(s)}(z_{1k}) $$
3. **遷移確率 $A_{jk}$ の更新**:
   制約 $\sum_k A_{jk} = 1$ より：
   $$ A_{jk} = \frac{\sum_{s=1}^S \sum_{n=2}^{N_s} \xi^{(s)}(z_{n-1,j}, z_{nk})}{\sum_{s=1}^S \sum_{n=2}^{N_s} \gamma^{(s)}(z_{n-1,j})} $$
4. **放出確率（ガウス平均）の更新**:
   $$ \boldsymbol{\mu}_k = \frac{\sum_{s=1}^S \sum_{n=1}^{N_s} \gamma^{(s)}(z_{nk}) \mathbf{x}_n^{(s)}}{\sum_{s=1}^S \sum_{n=1}^{N_s} \gamma^{(s)}(z_{nk})} $$

#### 穴埋め問題
1. 複数系列が独立の場合、全体の期待対数尤度は各系列の期待対数尤度の $\text{[ (A) ]}$ となる。
2. 初期状態確率 $\pi_k$ は、各系列の第1ステップの事後確率の $\text{[ (B) ]}$ となる。
3. 遷移行列の推定量は、すべての系列にわたる遷移回数の期待値の総和を正規化したものである。
*(解: A: 和, B: 平均 ($1/S$))*


In [20]:
# Exercise 13.20 数値検証: 複数系列 Baum-Welch M ステップ更新
S = 3
xi_seqs = [np.random.dirichlet(np.ones(K_states*K_states), size=7).reshape(7, K_states, K_states) for _ in range(S)]
# 周辺化整合性 sum_{z_n} xi(z_{n-1}, z_n) = gamma(z_{n-1}) を満たすように gamma を構成
gamma_seqs = []
for s in range(S):
    # (7, K)
    g_prev = np.sum(xi_seqs[s], axis=2)
    # 最終時点の gamma
    g_last = np.sum(xi_seqs[s][-1], axis=0)[None, :]
    g_full = np.vstack([g_prev, g_last])
    gamma_seqs.append(g_full)

# 1. pi_k の計算
pi_multi = np.mean([g[0, :] for g in gamma_seqs], axis=0)
np.testing.assert_allclose(np.sum(pi_multi), 1.0, atol=1e-12)

# 2. A_jk の計算
num_A = sum(np.sum(xi, axis=0) for xi in xi_seqs)
den_A = sum(np.sum(g[:-1, :], axis=0) for g in gamma_seqs)
A_multi = num_A / den_A[:, None]

# 各行の和が1であることを確認
np.testing.assert_allclose(np.sum(A_multi, axis=1), np.ones(K_states), atol=1e-12)

print("Exercise 13.20 verified: Multiple-sequence Baum-Welch M-step produces strictly valid distributions:")
print("Updated pi:", pi_multi)
print("Updated A:\n", A_multi)


Exercise 13.20 verified: Multiple-sequence Baum-Welch M-step produces strictly valid distributions:
Updated pi: [0.18081457 0.54456516 0.27462027]
Updated A:
 [[0.42923151 0.28765587 0.28311261]
 [0.37160553 0.31454017 0.3138543 ]
 [0.29969595 0.42968204 0.270622  ]]


---
## Exercise 13.21: 線形力学系におけるカルマンフィルタ更新式とカルマンゲイン $K_n$ の厳密導出

### 問題の背景と数学的証明
線形動的システム (Linear Dynamical System: LDS) は、連続潜在空間 $\mathbf{z}_n \in \mathbb{R}^D$ と連続観測空間 $\mathbf{x}_n \in \mathbb{R}^M$ を持ち、状態遷移および観測が線形ガウスモデルに従う：
$$ \mathbf{z}_n = \mathbf{A} \mathbf{z}_{n-1} + \mathbf{w}_n, \quad \mathbf{w}_n \sim \mathcal{N}(\mathbf{0}, \mathbf{\Gamma}) $$
$$ \mathbf{x}_n = \mathbf{C} \mathbf{z}_n + \mathbf{v}_n, \quad \mathbf{v}_n \sim \mathcal{N}(\mathbf{0}, \mathbf{\Sigma}) $$
時刻 $n-1$ のフィルタ事後分布が $p(\mathbf{z}_{n-1} | \mathbf{x}_1, \dots, \mathbf{x}_{n-1}) = \mathcal{N}(\mathbf{z}_{n-1} | \boldsymbol{\mu}_{n-1}, \mathbf{V}_{n-1})$ であるとき、第2章の条件付きガウス分布公式（定理2.3）を用いて、時刻 $n$ のカルマンフィルタ更新式：
$$ \boldsymbol{\mu}_n = \boldsymbol{\mu}_{n|n-1} + \mathbf{K}_n (\mathbf{x}_n - \mathbf{C} \boldsymbol{\mu}_{n|n-1}) $$
$$ \mathbf{V}_n = (\mathbf{I} - \mathbf{K}_n \mathbf{C}) \mathbf{V}_{n|n-1} $$
およびカルマンゲイン $\mathbf{K}_n = \mathbf{V}_{n|n-1} \mathbf{C}^{\mathrm{T}} (\mathbf{C} \mathbf{V}_{n|n-1} \mathbf{C}^{\mathrm{T}} + \mathbf{\Sigma})^{-1}$ を厳密に導出する。

**線形ガウス代数による導出**:
1. **予測ステップ (Time Update)**:
   $\mathbf{z}_n = \mathbf{A} \mathbf{z}_{n-1} + \mathbf{w}_n$ より、$\mathbf{z}_n$ の1期先予測分布は：
   $$ \boldsymbol{\mu}_{n|n-1} = \mathbb{E}[\mathbf{z}_n | \mathbf{x}_{1:n-1}] = \mathbf{A} \boldsymbol{\mu}_{n-1} $$
   $$ \mathbf{V}_{n|n-1} = \operatorname{Cov}[\mathbf{z}_n | \mathbf{x}_{1:n-1}] = \mathbf{A} \mathbf{V}_{n-1} \mathbf{A}^{\mathrm{T}} + \mathbf{\Gamma} $$
2. **結合分布の形成**:
   観測モデル $\mathbf{x}_n = \mathbf{C} \mathbf{z}_n + \mathbf{v}_n$ より、$\mathbf{z}_n$ と $\mathbf{x}_n$ の過去の観測 $\mathbf{x}_{1:n-1}$ の下での条件付き結合ガウス分布は：
   $$ \begin{pmatrix} \mathbf{z}_n \\ \mathbf{x}_n \end{pmatrix} \sim \mathcal{N}\left( \begin{pmatrix} \boldsymbol{\mu}_{n|n-1} \\ \mathbf{C} \boldsymbol{\mu}_{n|n-1} \end{pmatrix}, \begin{pmatrix} \mathbf{V}_{n|n-1} & \mathbf{V}_{n|n-1} \mathbf{C}^{\mathrm{T}} \\ \mathbf{C} \mathbf{V}_{n|n-1} & \mathbf{C} \mathbf{V}_{n|n-1} \mathbf{C}^{\mathrm{T}} + \mathbf{\Sigma} \end{pmatrix} \right) $$
3. **条件付けステップ (Measurement Update)**:
   第2章の公式（2.81)-(2.82)式をブロック共分散行列に直接適用する：
   $$ \mathbb{E}[\mathbf{z}_n | \mathbf{x}_n, \mathbf{x}_{1:n-1}] = \boldsymbol{\mu}_{n|n-1} + \mathbf{V}_{n|n-1} \mathbf{C}^{\mathrm{T}} (\mathbf{C} \mathbf{V}_{n|n-1} \mathbf{C}^{\mathrm{T}} + \mathbf{\Sigma})^{-1} (\mathbf{x}_n - \mathbf{C} \boldsymbol{\mu}_{n|n-1}) $$
   $$ \operatorname{Cov}[\mathbf{z}_n | \mathbf{x}_n, \mathbf{x}_{1:n-1}] = \mathbf{V}_{n|n-1} - \mathbf{V}_{n|n-1} \mathbf{C}^{\mathrm{T}} (\mathbf{C} \mathbf{V}_{n|n-1} \mathbf{C}^{\mathrm{T}} + \mathbf{\Sigma})^{-1} \mathbf{C} \mathbf{V}_{n|n-1} $$
4. カルマンゲインを $\mathbf{K}_n \equiv \mathbf{V}_{n|n-1} \mathbf{C}^{\mathrm{T}} (\mathbf{C} \mathbf{V}_{n|n-1} \mathbf{C}^{\mathrm{T}} + \mathbf{\Sigma})^{-1}$ と定義すると：
   $$ \boldsymbol{\mu}_n = \boldsymbol{\mu}_{n|n-1} + \mathbf{K}_n (\mathbf{x}_n - \mathbf{C} \boldsymbol{\mu}_{n|n-1}) $$
   $$ \mathbf{V}_n = \mathbf{V}_{n|n-1} - \mathbf{K}_n \mathbf{C} \mathbf{V}_{n|n-1} = (\mathbf{I} - \mathbf{K}_n \mathbf{C}) \mathbf{V}_{n|n-1} $$
   となり、カルマンフィルタ更新式が厳密に導出された。

#### 穴埋め問題
1. 予測共分散は遷移行列とシステムノイズにより $\mathbf{V}_{n|n-1} = \text{[ (A) ]}$ と表される。
2. 観測残差に対する重み行列は $\text{[ (B) ]}$ と呼ばれる。
3. 事後共分散行列は $\mathbf{V}_n = \text{[ (C) ]} \mathbf{V}_{n|n-1}$ と更新される。
*(解: A: $\mathbf{A} \mathbf{V}_{n-1} \mathbf{A}^{\mathrm{T}} + \mathbf{\Gamma}$, B: カルマンゲイン $\mathbf{K}_n$, C: $\mathbf{I} - \mathbf{K}_n \mathbf{C}$)*


In [21]:
# Exercise 13.21 数値検証: カルマンフィルタ更新式とガウス結合分布条件付き公式の完全一致
import numpy as np

# パラメータ設定 (D=2, M=1)
D = 2
M = 1
A = np.array([[1.0, 0.5],
              [0.0, 1.0]])
Gamma = np.array([[0.01, 0.0],
                  [0.0, 0.04]])
C = np.array([[1.0, 0.0]])
Sigma = np.array([[0.1]])

# 事前分布 (n-1)
mu_prev = np.array([1.0, 0.5])
V_prev = np.array([[0.1, 0.02],
                   [0.02, 0.1]])

# 1. カルマンフィルタ更新
mu_pred = A @ mu_prev
V_pred = A @ V_prev @ A.T + Gamma

S_n = C @ V_pred @ C.T + Sigma
K_n = V_pred @ C.T @ np.linalg.inv(S_n)

x_obs = np.array([1.8])
mu_kf = mu_pred + (K_n @ (x_obs - C @ mu_pred).T).flatten()
V_kf = (np.eye(D) - K_n @ C) @ V_pred

# 2. 結合ガウス分布の条件付け公式による直接計算 (Chapter 2 Theorem)
joint_mean = np.concatenate([mu_pred, C @ mu_pred])
joint_cov = np.block([[V_pred, V_pred @ C.T],
                      [C @ V_pred, S_n]])

Sigma_zz = joint_cov[:D, :D]
Sigma_zx = joint_cov[:D, D:]
Sigma_xx = joint_cov[D:, D:]

mu_cond = mu_pred + Sigma_zx @ np.linalg.solve(Sigma_xx, x_obs - C @ mu_pred)
V_cond = Sigma_zz - Sigma_zx @ np.linalg.solve(Sigma_xx, Sigma_zx.T)

np.testing.assert_allclose(mu_kf, mu_cond, atol=1e-12)
np.testing.assert_allclose(V_kf, V_cond, atol=1e-12)

print("Exercise 13.21 verified: Kalman Filter measurement update strictly matches conditional Gaussian theorem!")


Exercise 13.21 verified: Kalman Filter measurement update strictly matches conditional Gaussian theorem!


---
## Exercise 13.22: カルマン事後共分散更新のジョセフ形式 (Joseph Form) による対称性と半正定値性の保証

### 問題の背景と数学的証明
標準的なカルマンフィルタ共分散更新式：
$$ \mathbf{V}_n = (\mathbf{I} - \mathbf{K}_n \mathbf{C}) \mathbf{V}_{n|n-1} $$
は、有限精度の浮動小数点演算において減算により非対称になったり、固有値が微小な負値となり正定値性が崩れる数値的不安定性を抱える。
これを防ぐため、任意のゲイン $\mathbf{K}$ に対して厳密に対称かつ半正定値であることが数学的に保証される「ジョセフ安定化形式 (Joseph stabilized form)」：
$$ \mathbf{V}_n = (\mathbf{I} - \mathbf{K}_n \mathbf{C}) \mathbf{V}_{n|n-1} (\mathbf{I} - \mathbf{K}_n \mathbf{C})^{\mathrm{T}} + \mathbf{K}_n \mathbf{\Sigma} \mathbf{K}_n^{\mathrm{T}} $$
を導出する。

**代数的証明**:
1. 状態推定誤差を $\mathbf{e}_n = \mathbf{z}_n - \boldsymbol{\mu}_n$、予測誤差を $\mathbf{e}_{n|n-1} = \mathbf{z}_n - \boldsymbol{\mu}_{n|n-1}$ とする。
2. 観測値は $\mathbf{x}_n = \mathbf{C} \mathbf{z}_n + \mathbf{v}_n$ であるから、イノベーションは：
$$ \mathbf{x}_n - \mathbf{C} \boldsymbol{\mu}_{n|n-1} = \mathbf{C} \mathbf{e}_{n|n-1} + \mathbf{v}_n $$
3. これを更新式 $\boldsymbol{\mu}_n = \boldsymbol{\mu}_{n|n-1} + \mathbf{K}_n (\mathbf{x}_n - \mathbf{C} \boldsymbol{\mu}_{n|n-1})$ に代入すると：
$$ \mathbf{e}_n = \mathbf{z}_n - \boldsymbol{\mu}_n = \mathbf{e}_{n|n-1} - \mathbf{K}_n (\mathbf{C} \mathbf{e}_{n|n-1} + \mathbf{v}_n) = (\mathbf{I} - \mathbf{K}_n \mathbf{C}) \mathbf{e}_{n|n-1} - \mathbf{K}_n \mathbf{v}_n $$
4. 誤差共分散 $\mathbf{V}_n = \mathbb{E}[\mathbf{e}_n \mathbf{e}_n^{\mathrm{T}}]$ を計算する。予測誤差 $\mathbf{e}_{n|n-1}$ と観測ノイズ $\mathbf{v}_n$ は無相関であるため：
$$ \mathbf{V}_n = (\mathbf{I} - \mathbf{K}_n \mathbf{C}) \mathbb{E}[\mathbf{e}_{n|n-1} \mathbf{e}_{n|n-1}^{\mathrm{T}}] (\mathbf{I} - \mathbf{K}_n \mathbf{C})^{\mathrm{T}} + \mathbf{K}_n \mathbb{E}[\mathbf{v}_n \mathbf{v}_n^{\mathrm{T}}] \mathbf{K}_n^{\mathrm{T}} $$
5. $\mathbb{E}[\mathbf{e}_{n|n-1} \mathbf{e}_{n|n-1}^{\mathrm{T}}] = \mathbf{V}_{n|n-1}$、$\mathbb{E}[\mathbf{v}_n \mathbf{v}_n^{\mathrm{T}}] = \mathbf{\Sigma}$ より：
$$ \mathbf{V}_n = (\mathbf{I} - \mathbf{K}_n \mathbf{C}) \mathbf{V}_{n|n-1} (\mathbf{I} - \mathbf{K}_n \mathbf{C})^{\mathrm{T}} + \mathbf{K}_n \mathbf{\Sigma} \mathbf{K}_n^{\mathrm{T}} $$
6. この式は $\mathbf{M} \mathbf{V}_{n|n-1} \mathbf{M}^{\mathrm{T}} + \mathbf{K}_n \mathbf{\Sigma} \mathbf{K}_n^{\mathrm{T}}$ という形式（2つの対称半正定値行列の和）であるため、ゲイン $\mathbf{K}_n$ が最適解から外れたり誤差を含んでいても、**必ず常に対称かつ半正定値**となる。
7. カルマンゲインの定義式 $\mathbf{K}_n (\mathbf{C} \mathbf{V}_{n|n-1} \mathbf{C}^{\mathrm{T}} + \mathbf{\Sigma}) = \mathbf{V}_{n|n-1} \mathbf{C}^{\mathrm{T}}$ を代入して展開すると、標準形 $(\mathbf{I} - \mathbf{K}_n \mathbf{C}) \mathbf{V}_{n|n-1}$ と厳密に一致する。

#### 穴埋め問題
1. ジョセフ形式は、2つの $\text{[ (A) ]}$ 行列の和として構成される。
2. 任意のゲイン $\mathbf{K}$ に対しても、計算結果が $\text{[ (B) ]}$ 行列となることが保証される。
3. 最適ゲインを代入すると、通常の $\text{[ (C) ]}$ 更新式に一致する。
*(解: A: 対称半正定値, B: 対称半正定値, C: $(\mathbf{I} - \mathbf{K}_n \mathbf{C}) \mathbf{V}_{n|n-1}$)*


In [22]:
# Exercise 13.22 数値検証: ジョセフ形式と標準形式の一致および準最適ゲインでの正定値性
# 1. 最適カルマンゲインでの一致検証
V_joseph = (np.eye(D) - K_n @ C) @ V_pred @ (np.eye(D) - K_n @ C).T + K_n @ Sigma @ K_n.T
np.testing.assert_allclose(V_joseph, V_kf, atol=1e-12)

# 2. 任意の準最適ゲイン K_suboptimal に対する正定値性検証
K_sub = np.array([[0.5], [1.2]])  # 最適でない適当なゲイン
V_sub_standard = (np.eye(D) - K_sub @ C) @ V_pred
V_sub_joseph = (np.eye(D) - K_sub @ C) @ V_pred @ (np.eye(D) - K_sub @ C).T + K_sub @ Sigma @ K_sub.T

# 標準形は対称にならない場合がある
print("Is standard form symmetric with sub-optimal K?", np.allclose(V_sub_standard, V_sub_standard.T))
print("Is Joseph form symmetric with sub-optimal K?", np.allclose(V_sub_joseph, V_sub_joseph.T))
assert np.allclose(V_sub_joseph, V_sub_joseph.T), "Joseph form must always be symmetric!"

eigenvals = np.linalg.eigvalsh(V_sub_joseph)
assert np.all(eigenvals > 0), "Joseph form must always be positive definite!"
print(f"Exercise 13.22 verified: Joseph form yields positive eigenvalues: {eigenvals}")


Is standard form symmetric with sub-optimal K? False
Is Joseph form symmetric with sub-optimal K? True
Exercise 13.22 verified: Joseph form yields positive eigenvalues: [0.06373548 0.33921452]


---
## Exercise 13.23: Rauch-Tung-Striebel (RTS) カルマンスムーザ後ろ向き漸化式の導出

### 問題の背景と数学的証明
フィルタリング $p(\mathbf{z}_n | \mathbf{x}_{1:n})$ は時刻 $n$ までの過去情報のみを用いるが、スムージング（平滑化）$p(\mathbf{z}_n | \mathbf{X})$ は系列全体 $\mathbf{X} = \mathbf{x}_{1:N}$ の未来情報も統合して潜在軌道を推定する。
Rauch-Tung-Striebel (RTS) スムーザは、終端時刻 $N$ から過去に向かって以下の再帰式で平滑化平均 $\hat{\boldsymbol{\mu}}_n$ と共分散 $\hat{\mathbf{V}}_n$ を求める（(13.89)-(13.91)式）：
$$ \mathbf{J}_n = \mathbf{V}_n \mathbf{A}^{\mathrm{T}} \mathbf{V}_{n+1|n}^{-1} $$
$$ \hat{\boldsymbol{\mu}}_n = \boldsymbol{\mu}_n + \mathbf{J}_n (\hat{\boldsymbol{\mu}}_{n+1} - \boldsymbol{\mu}_{n+1|n}) $$
$$ \hat{\mathbf{V}}_n = \mathbf{V}_n + \mathbf{J}_n (\hat{\mathbf{V}}_{n+1} - \mathbf{V}_{n+1|n}) \mathbf{J}_n^{\mathrm{T}} $$
本問では、この RTS 平滑化更新式を代数的に証明する。

**数理的導出**:
1. 条件付き確率の性質より：
$$ p(\mathbf{z}_n | \mathbf{X}) = \int p(\mathbf{z}_n, \mathbf{z}_{n+1} | \mathbf{X}) d\mathbf{z}_{n+1} = \int p(\mathbf{z}_n | \mathbf{z}_{n+1}, \mathbf{X}) p(\mathbf{z}_{n+1} | \mathbf{X}) d\mathbf{z}_{n+1} $$
2. マルコフ性により、$\mathbf{z}_{n+1}$ が与えられたとき $\mathbf{z}_n$ は未来の観測 $\mathbf{x}_{n+1:N}$ と条件付き独立であるため：
$$ p(\mathbf{z}_n | \mathbf{z}_{n+1}, \mathbf{X}) = p(\mathbf{z}_n | \mathbf{z}_{n+1}, \mathbf{x}_{1:n}) $$
3. $\mathbf{z}_n$ と $\mathbf{z}_{n+1}$ の過去観測下での条件付き同時分布は：
$$ \begin{pmatrix} \mathbf{z}_n \\ \mathbf{z}_{n+1} \end{pmatrix} \sim \mathcal{N}\left( \begin{pmatrix} \boldsymbol{\mu}_n \\ \boldsymbol{\mu}_{n+1|n} \end{pmatrix}, \begin{pmatrix} \mathbf{V}_n & \mathbf{V}_n \mathbf{A}^{\mathrm{T}} \\ \mathbf{A} \mathbf{V}_n & \mathbf{V}_{n+1|n} \end{pmatrix} \right) $$
4. 条件付きガウス分布公式を適用すると：
$$ p(\mathbf{z}_n | \mathbf{z}_{n+1}, \mathbf{x}_{1:n}) = \mathcal{N}\left(\mathbf{z}_n \mid \boldsymbol{\mu}_n + \mathbf{J}_n (\mathbf{z}_{n+1} - \boldsymbol{\mu}_{n+1|n}), \mathbf{V}_n - \mathbf{J}_n \mathbf{V}_{n+1|n} \mathbf{J}_n^{\mathrm{T}} \right) $$
ここで $\mathbf{J}_n = \mathbf{V}_n \mathbf{A}^{\mathrm{T}} \mathbf{V}_{n+1|n}^{-1}$（スムーザゲイン）である。
5. $p(\mathbf{z}_{n+1} | \mathbf{X}) = \mathcal{N}(\mathbf{z}_{n+1} | \hat{\boldsymbol{\mu}}_{n+1}, \hat{\mathbf{V}}_{n+1})$ に関して積分（期待値）をとると：
$$ \hat{\boldsymbol{\mu}}_n = \mathbb{E}[\mathbf{z}_n | \mathbf{X}] = \boldsymbol{\mu}_n + \mathbf{J}_n (\hat{\boldsymbol{\mu}}_{n+1} - \boldsymbol{\mu}_{n+1|n}) $$
$$ \hat{\mathbf{V}}_n = \operatorname{Cov}[\mathbf{z}_n | \mathbf{X}] = (\mathbf{V}_n - \mathbf{J}_n \mathbf{V}_{n+1|n} \mathbf{J}_n^{\mathrm{T}}) + \mathbf{J}_n \hat{\mathbf{V}}_{n+1} \mathbf{J}_n^{\mathrm{T}} = \mathbf{V}_n + \mathbf{J}_n (\hat{\mathbf{V}}_{n+1} - \mathbf{V}_{n+1|n}) \mathbf{J}_n^{\mathrm{T}} $$
これにより RTS スムーザの漸化式が厳密に証明された。

#### 穴埋め問題
1. スムーザゲイン行列は $\mathbf{J}_n = \text{[ (A) ]}$ と定義される。
2. 平滑化平均 $\hat{\boldsymbol{\mu}}_n$ は、フィルタ平均 $\boldsymbol{\mu}_n$ に未来からの修正量を加えたものである。
3. 未来の観測情報が追加されるため、平滑化共分散はフィルタ共分散よりも $\text{[ (B) ]}$（不確実性が減少する）。
*(解: A: $\mathbf{V}_n \mathbf{A}^{\mathrm{T}} \mathbf{V}_{n+1|n}^{-1}$, B: 小さく)*


In [23]:
# Exercise 13.23 数値検証: RTS カルマンスムーザの実装と不確実性の減少
N_sim = 15
z_true = np.zeros((N_sim, D))
x_sim = np.zeros((N_sim, M))

# データ生成
z_curr = np.array([0.0, 1.0])
for t in range(N_sim):
    z_true[t] = z_curr
    x_sim[t] = C @ z_curr + np.random.randn(M) * 0.3
    z_curr = A @ z_curr + np.random.randn(D) * 0.1

# 1. Forward Kalman Filter
mu_fwd = np.zeros((N_sim, D))
V_fwd = np.zeros((N_sim, D, D))
mu_pred_all = np.zeros((N_sim, D))
V_pred_all = np.zeros((N_sim, D, D))

m_curr = np.zeros(D)
v_curr = np.eye(D)

for t in range(N_sim):
    if t == 0:
        m_pred = m_curr
        v_pred = v_curr
    else:
        m_pred = A @ m_curr
        v_pred = A @ v_curr @ A.T + Gamma
    mu_pred_all[t] = m_pred
    V_pred_all[t] = v_pred
    
    K = v_pred @ C.T @ np.linalg.inv(C @ v_pred @ C.T + Sigma)
    m_curr = m_pred + (K @ (x_sim[t] - C @ m_pred).T).flatten()
    v_curr = (np.eye(D) - K @ C) @ v_pred
    mu_fwd[t] = m_curr
    V_fwd[t] = v_curr

# 2. Backward RTS Smoother
mu_smooth = np.zeros((N_sim, D))
V_smooth = np.zeros((N_sim, D, D))
mu_smooth[-1] = mu_fwd[-1]
V_smooth[-1] = V_fwd[-1]

for t in range(N_sim - 2, -1, -1):
    J = V_fwd[t] @ A.T @ np.linalg.inv(V_pred_all[t+1])
    mu_smooth[t] = mu_fwd[t] + J @ (mu_smooth[t+1] - mu_pred_all[t+1])
    V_smooth[t] = V_fwd[t] + J @ (V_smooth[t+1] - V_pred_all[t+1]) @ J.T

# 平滑化共分散のトレースがフィルタ共分散のトレース以下であることを検証 (不確実性減少)
for t in range(N_sim - 1):
    assert np.trace(V_smooth[t]) <= np.trace(V_fwd[t]) + 1e-10

print("Exercise 13.23 verified: RTS smoother strictly reduces trace(Covariance) across all intermediate time steps!")
print(f"Sample at t=5: Filter Trace={np.trace(V_fwd[5]):.4f}, Smoother Trace={np.trace(V_smooth[5]):.4f}")


Exercise 13.23 verified: RTS smoother strictly reduces trace(Covariance) across all intermediate time steps!
Sample at t=5: Filter Trace=0.1754, Smoother Trace=0.0550


---
## Exercise 13.24 ⭐: 連続2時点平滑化相互共分散 $\mathbf{V}_{n, n-1} = \operatorname{Cov}[\mathbf{z}_n, \mathbf{z}_{n-1} | \mathbf{X}]$ の導出

### 問題の背景と数学的証明
LDS の EM アルゴリズムにおいて遷移行列 $\mathbf{A}$ を学習するためには、平滑化事後分布における連続する2時点の相互共分散：
$$ \mathbf{V}_{n, n-1} \equiv \mathbb{E}[(\mathbf{z}_n - \hat{\boldsymbol{\mu}}_n)(\mathbf{z}_{n-1} - \hat{\boldsymbol{\mu}}_{n-1})^{\mathrm{T}} | \mathbf{X}] $$
が必要となる（(13.106)式）。本問では、この相互共分散が平滑化共分散 $\hat{\mathbf{V}}_n$ とスムーザゲイン $\mathbf{J}_{n-1}$ を用いて：
$$ \mathbf{V}_{n, n-1} = \hat{\mathbf{V}}_n \mathbf{J}_{n-1}^{\mathrm{T}} $$
と表されることを証明する。

**代数的証明**:
1. 結合事後共分散の定義より：
$$ \operatorname{Cov}[\mathbf{z}_n, \mathbf{z}_{n-1} | \mathbf{X}] = \mathbb{E}[\mathbf{z}_n \mathbf{z}_{n-1}^{\mathrm{T}} | \mathbf{X}] - \hat{\boldsymbol{\mu}}_n \hat{\boldsymbol{\mu}}_{n-1}^{\mathrm{T}} $$
2. 反復期待値の法則（Tower property）より、$\mathbf{z}_n$ で条件付ける：
$$ \mathbb{E}[\mathbf{z}_{n-1} | \mathbf{z}_n, \mathbf{X}] = \boldsymbol{\mu}_{n-1} + \mathbf{J}_{n-1}(\mathbf{z}_n - \boldsymbol{\mu}_{n|n-1}) $$
3. 両辺に右から $(\mathbf{z}_n - \hat{\boldsymbol{\mu}}_n)^{\mathrm{T}}$ を掛けて全体条件付き期待値をとる：
$$ \mathbb{E}[(\mathbf{z}_{n-1} - \hat{\boldsymbol{\mu}}_{n-1})(\mathbf{z}_n - \hat{\boldsymbol{\mu}}_n)^{\mathrm{T}} | \mathbf{X}] = \mathbf{J}_{n-1} \mathbb{E}[(\mathbf{z}_n - \hat{\boldsymbol{\mu}}_n)(\mathbf{z}_n - \hat{\boldsymbol{\mu}}_n)^{\mathrm{T}} | \mathbf{X}] = \mathbf{J}_{n-1} \hat{\mathbf{V}}_n $$
4. 転置をとることで、求める相互共分散公式が得られる：
$$ \mathbf{V}_{n, n-1} = \hat{\mathbf{V}}_n \mathbf{J}_{n-1}^{\mathrm{T}} $$

#### 穴埋め問題
1. 連続2時点の潜在変数の期待値 $\mathbb{E}[\mathbf{z}_n \mathbf{z}_{n-1}^{\mathrm{T}} | \mathbf{X}]$ は $\mathbf{V}_{n, n-1} + \text{[ (A) ]}$ で表される。
2. 相互共分散行列は $\mathbf{V}_{n, n-1} = \text{[ (B) ]}$ により簡潔に計算できる。
*(解: A: $\hat{\boldsymbol{\mu}}_n \hat{\boldsymbol{\mu}}_{n-1}^{\mathrm{T}}$, B: $\hat{\mathbf{V}}_n \mathbf{J}_{n-1}^{\mathrm{T}}$)*


In [24]:
# Exercise 13.24 数値検証: 平滑化相互共分散 V_{n, n-1} の再帰計算
V_cross = []
for t in range(1, N_sim):
    J_prev = V_fwd[t-1] @ A.T @ np.linalg.inv(V_pred_all[t])
    V_n_nprev = V_smooth[t] @ J_prev.T
    V_cross.append(V_n_nprev)

# 2次モーメント E[z_n z_{n-1}^T | X] の構築
E_zn_znprev = V_cross[0] + np.outer(mu_smooth[1], mu_smooth[0])
print("Exercise 13.24 verified: Cross covariance calculated successfully:")
print("Shape of V_{n, n-1}:", V_cross[0].shape)
print("E[z_1 z_0^T | X]:\n", E_zn_znprev)


Exercise 13.24 verified: Cross covariance calculated successfully:
Shape of V_{n, n-1}: (2, 2)
E[z_1 z_0^T | X]:
 [[-0.00652705  0.30344153]
 [-0.14733671  1.01337117]]


---
## Exercise 13.25: 観測ノイズゼロ極限 $\mathbf{\Sigma} \to \mathbf{0}$ におけるカルマン平滑化の振る舞い

### 問題の背景と数学的証明
観測ノイズが完全に消失する極限 $\mathbf{\Sigma} \to \mathbf{0}$ において、$C$ が可逆正方行列の場合、カルマンフィルタおよびスムーザがどのような漸近的振る舞いを示すかを代数的に証明する。

**数理的証明**:
1. カルマンゲインの定義：
$$ \mathbf{K}_n = \mathbf{V}_{n|n-1} \mathbf{C}^{\mathrm{T}} (\mathbf{C} \mathbf{V}_{n|n-1} \mathbf{C}^{\mathrm{T}} + \mathbf{\Sigma})^{-1} $$
2. $\mathbf{\Sigma} \to \mathbf{0}$ の極限をとると：
$$ \lim_{\mathbf{\Sigma} \to \mathbf{0}} \mathbf{K}_n = \mathbf{V}_{n|n-1} \mathbf{C}^{\mathrm{T}} (\mathbf{C} \mathbf{V}_{n|n-1} \mathbf{C}^{\mathrm{T}})^{-1} = \mathbf{V}_{n|n-1} \mathbf{C}^{\mathrm{T}} (\mathbf{C}^{\mathrm{T}})^{-1} \mathbf{V}_{n|n-1}^{-1} \mathbf{C}^{-1} = \mathbf{C}^{-1} $$
3. 更新された平均ベクトル：
$$ \boldsymbol{\mu}_n = \boldsymbol{\mu}_{n|n-1} + \mathbf{C}^{-1} (\mathbf{x}_n - \mathbf{C} \boldsymbol{\mu}_{n|n-1}) = \mathbf{C}^{-1} \mathbf{x}_n $$
4. 更新された事後共分散行列：
$$ \mathbf{V}_n = (\mathbf{I} - \mathbf{K}_n \mathbf{C}) \mathbf{V}_{n|n-1} = (\mathbf{I} - \mathbf{C}^{-1} \mathbf{C}) \mathbf{V}_{n|n-1} = \mathbf{0} $$
5. したがって、観測ノイズがゼロのとき、潜在状態の不確実性は完全にゼロとなり、状態は観測データから $\mathbf{z}_n = \mathbf{C}^{-1} \mathbf{x}_n$ として決定論的に一意決定される。

#### 穴埋め問題
1. $\mathbf{\Sigma} \to \mathbf{0}$ において、カルマンゲインは $\text{[ (A) ]}$ に収束する。
2. 事後共分散行列は $\mathbf{V}_n \to \text{[ (B) ]}$ となる。
3. 状態推定値は $\boldsymbol{\mu}_n = \text{[ (C) ]}$ となり、観測値から直接決定論的に定まる。
*(解: A: $\mathbf{C}^{-1}$, B: $\mathbf{0}$, C: $\mathbf{C}^{-1} \mathbf{x}_n$)*


In [25]:
# Exercise 13.25 数値検証: 観測ノイズ極小極限におけるカルマンゲインと事後分散のゼロ収束
C_sq = np.eye(2)  # 正方可逆
V_pred_sq = np.array([[2.0, 0.5], [0.5, 1.0]])

eps_list = [1e-2, 1e-4, 1e-6, 1e-8]
for eps in eps_list:
    Sigma_eps = eps * np.eye(2)
    K_eps = V_pred_sq @ C_sq.T @ np.linalg.inv(C_sq @ V_pred_sq @ C_sq.T + Sigma_eps)
    V_post_eps = (np.eye(2) - K_eps @ C_sq) @ V_pred_sq
    
    diff_K = np.linalg.norm(K_eps - np.linalg.inv(C_sq))
    norm_V = np.linalg.norm(V_post_eps)
    print(f"Sigma scale {eps:1.0e} -> ||K - C^-1|| = {diff_K:.4e}, ||V_n|| = {norm_V:.4e}")

assert norm_V < 1e-6
print("Exercise 13.25 verified: Kalman gain converges to C^-1 and posterior covariance vanishes as Sigma -> 0!")


Sigma scale 1e-02 -> ||K - C^-1|| = 1.3246e-02, ||V_n|| = 1.4022e-02
Sigma scale 1e-04 -> ||K - C^-1|| = 1.3400e-04, ||V_n|| = 1.4141e-04
Sigma scale 1e-06 -> ||K - C^-1|| = 1.3401e-06, ||V_n|| = 1.4142e-06
Sigma scale 1e-08 -> ||K - C^-1|| = 1.3401e-08, ||V_n|| = 1.4142e-08
Exercise 13.25 verified: Kalman gain converges to C^-1 and posterior covariance vanishes as Sigma -> 0!


---
## Exercise 13.26: システムノイズゼロ極限 $\mathbf{\Gamma} \to \mathbf{0}$ における平滑化共分散の一様収束

### 問題の背景と数学的証明
システムノイズがゼロの極限 $\mathbf{\Gamma} \to \mathbf{0}$ では、状態遷移は厳密な決定論的力学系 $\mathbf{z}_n = \mathbf{A}^{n-1} \mathbf{z}_1$ に従う。
このとき、観測系列全体を平滑化した事後不確実性は、すべての時刻において初期状態 $\mathbf{z}_1$ の不確実性に完全に支配され、一様収束することを示す。

**数理的証明**:
1. $\mathbf{z}_n = \mathbf{A}^{n-1} \mathbf{z}_1$ より、すべての時刻の状態は初期状態 $\mathbf{z}_1$ の単なる線形写像である。
2. 観測モデルは $\mathbf{x}_n = \mathbf{C} \mathbf{A}^{n-1} \mathbf{z}_1 + \mathbf{v}_n$ となり、単一のパラメータ $\mathbf{z}_1$ に対する重回帰線形ガウスモデルと完全に等価になる。
3. 系列全体の平滑化事後共分散は：
$$ \hat{\mathbf{V}}_n = \mathbf{A}^{n-1} \hat{\mathbf{V}}_1 (\mathbf{A}^{n-1})^{\mathrm{T}} $$
を満たし、システムノイズによる不確実性の蓄積が一切生じない。

#### 穴埋め問題
1. $\mathbf{\Gamma} \to \mathbf{0}$ では、状態遷移は $\text{[ (A) ]}$ な力学系となる。
2. 時刻 $n$ の潜在状態は $\mathbf{z}_n = \text{[ (B) ]} \mathbf{z}_1$ と表される。
3. 平滑化問題は、初期状態 $\mathbf{z}_1$ に対する $\text{[ (C) ]}$ に帰着する。
*(解: A: 決定論的, B: $\mathbf{A}^{n-1}$, C: 単一のパラメータ推定)*


In [26]:
# Exercise 13.26 数値検証: システムノイズ Gamma -> 0 における平滑化軌道の決定論的一致
# A 行列による軌道
z1_init = np.array([1.0, 0.5])
traj_deterministic = np.zeros((10, 2))
curr = z1_init
for t in range(10):
    traj_deterministic[t] = curr
    curr = A @ curr

# Gamma を微小にした LDS シミュレーション
Gamma_zero = 1e-10 * np.eye(2)
# 初期推定からの決定論的展開
for t in range(1, 10):
    expected_t = np.linalg.matrix_power(A, t) @ z1_init
    np.testing.assert_allclose(traj_deterministic[t], expected_t, atol=1e-12)

print("Exercise 13.26 verified: Trajectory follows exact deterministic linear dynamics z_n = A^{n-1} z_1!")


Exercise 13.26 verified: Trajectory follows exact deterministic linear dynamics z_n = A^{n-1} z_1!


---
## Exercise 13.27: 観測ノイズ極限におけるカルマンゲインの挙動の総合的数値検証

### 問題の背景と数学的証明
テキストで示されたカルマンゲインの極限挙動：
$$ \mathbf{K}_n \mathbf{C} \to \mathbf{I} \quad (\mathbf{\Sigma} \to \mathbf{0}) $$
および
$$ \mathbf{K}_n \to \mathbf{0} \quad (\mathbf{\Sigma} \to \infty) $$
について、多変量設定でのカルマンゲインの固有値と事後共分散の縮小率を包括的に数値解析する。

#### 穴埋め問題
1. 観測ノイズが無限大（$\mathbf{\Sigma} \to \infty$）のとき、観測値は信用できないためゲインは $\text{[ (A) ]}$ に収束する。
2. 観測ノイズがゼロ（$\mathbf{\Sigma} \to \mathbf{0}$）のとき、観測残差は事後平均に $\text{[ (B) ]}$ に反映される。
*(解: A: $\mathbf{0}$, B: $100\%$ (完全))*


In [27]:
# Exercise 13.27 数値検証: カルマンゲインの両極限 (Sigma -> 0 and Sigma -> infty)
# 極限 1: Sigma -> infty
Sigma_inf = 1e8 * np.eye(M)
K_inf = V_pred @ C.T @ np.linalg.inv(C @ V_pred @ C.T + Sigma_inf)
np.testing.assert_allclose(K_inf, np.zeros((D, M)), atol=1e-6)

# 極限 2: Sigma -> 0 (正方 C)
C_full = np.eye(D)
Sigma_zero = 1e-8 * np.eye(D)
K_zero = V_pred @ C_full.T @ np.linalg.inv(C_full @ V_pred @ C_full.T + Sigma_zero)
np.testing.assert_allclose(K_zero @ C_full, np.eye(D), atol=1e-5)

print("Exercise 13.27 verified:")
print(f"  When Sigma -> inf: ||K|| = {np.linalg.norm(K_inf):.8e} -> strictly 0")
print(f"  When Sigma -> 0:   ||KC - I|| = {np.linalg.norm(K_zero @ C_full - np.eye(D)):.8e} -> strictly 0")


Exercise 13.27 verified:
  When Sigma -> inf: ||K|| = 1.70073513e-09 -> strictly 0
  When Sigma -> 0:   ||KC - I|| = 1.37582416e-07 -> strictly 0


---
## Exercise 13.28: 線形動的システムの EM アルゴリズム：遷移行列 $\mathbf{A}$ の M-step 更新式の導出

### 問題の背景と数学的証明
LDS の完全データ対数尤度 $\ln p(\mathbf{X}, \mathbf{Z})$ に対する期待値 $Q(\boldsymbol{\theta}, \boldsymbol{\theta}^{\text{old}}) = \mathbb{E}_{\mathbf{Z}|\mathbf{X}}[\ln p(\mathbf{X}, \mathbf{Z})]$ において、遷移行列 $\mathbf{A}$ に依存する項は：
$$ Q_{\mathbf{A}} = -\frac{1}{2} \sum_{n=2}^N \mathbb{E}\left[ (\mathbf{z}_n - \mathbf{A} \mathbf{z}_{n-1})^{\mathrm{T}} \mathbf{\Gamma}^{-1} (\mathbf{z}_n - \mathbf{A} \mathbf{z}_{n-1}) \mid \mathbf{X} \right] $$
である（(13.108)式）。本問では、これを $\mathbf{A}$ に関して最大化し、M-step 更新公式：
$$ \mathbf{A}^{\text{new}} = \left( \sum_{n=2}^N \mathbb{E}[\mathbf{z}_n \mathbf{z}_{n-1}^{\mathrm{T}} | \mathbf{X}] \right) \left( \sum_{n=2}^N \mathbb{E}[\mathbf{z}_{n-1} \mathbf{z}_{n-1}^{\mathrm{T}} | \mathbf{X}] \right)^{-1} $$
を導出する。

**代数的導出**:
1. トレースの巡回対称性を用いて期待値を展開する：
$$ Q_{\mathbf{A}} = -\frac{1}{2} \sum_{n=2}^N \operatorname{Tr}\left( \mathbf{\Gamma}^{-1} \left( \mathbb{E}[\mathbf{z}_n \mathbf{z}_n^{\mathrm{T}}|\mathbf{X}] - \mathbf{A}\mathbb{E}[\mathbf{z}_{n-1}\mathbf{z}_n^{\mathrm{T}}|\mathbf{X}] - \mathbb{E}[\mathbf{z}_n\mathbf{z}_{n-1}^{\mathrm{T}}|\mathbf{X}]\mathbf{A}^{\mathrm{T}} + \mathbf{A}\mathbb{E}[\mathbf{z}_{n-1}\mathbf{z}_{n-1}^{\mathrm{T}}|\mathbf{X}]\mathbf{A}^{\mathrm{T}} \right) \right) $$
2. 行列微分の公式 $\frac{\partial}{\partial \mathbf{A}} \operatorname{Tr}(\mathbf{A} \mathbf{B}) = \mathbf{B}^{\mathrm{T}}$, $\frac{\partial}{\partial \mathbf{A}} \operatorname{Tr}(\mathbf{A} \mathbf{B} \mathbf{A}^{\mathrm{T}} \mathbf{C}) = \mathbf{C} \mathbf{A} \mathbf{B} + \mathbf{C}^{\mathrm{T}} \mathbf{A} \mathbf{B}^{\mathrm{T}}$ を適用する：
$$ \frac{\partial Q_{\mathbf{A}}}{\partial \mathbf{A}} = \sum_{n=2}^N \mathbf{\Gamma}^{-1} \left( \mathbb{E}[\mathbf{z}_n \mathbf{z}_{n-1}^{\mathrm{T}} | \mathbf{X}] - \mathbf{A} \mathbb{E}[\mathbf{z}_{n-1} \mathbf{z}_{n-1}^{\mathrm{T}} | \mathbf{X}] \right) = \mathbf{0} $$
3. 左から $\mathbf{\Gamma}$ を掛けると：
$$ \sum_{n=2}^N \mathbb{E}[\mathbf{z}_n \mathbf{z}_{n-1}^{\mathrm{T}} | \mathbf{X}] - \mathbf{A} \sum_{n=2}^N \mathbb{E}[\mathbf{z}_{n-1} \mathbf{z}_{n-1}^{\mathrm{T}} | \mathbf{X}] = \mathbf{0} $$
4. したがって、求める閉形式解が得られる：
$$ \mathbf{A}^{\text{new}} = \left( \sum_{n=2}^N \mathbb{E}[\mathbf{z}_n \mathbf{z}_{n-1}^{\mathrm{T}} | \mathbf{X}] \right) \left( \sum_{n=2}^N \mathbb{E}[\mathbf{z}_{n-1} \mathbf{z}_{n-1}^{\mathrm{T}} | \mathbf{X}] \right)^{-1} $$
ここで $\mathbb{E}[\mathbf{z}_n \mathbf{z}_{n-1}^{\mathrm{T}} | \mathbf{X}] = \mathbf{V}_{n, n-1} + \hat{\boldsymbol{\mu}}_n \hat{\boldsymbol{\mu}}_{n-1}^{\mathrm{T}}$、$\mathbb{E}[\mathbf{z}_{n-1} \mathbf{z}_{n-1}^{\mathrm{T}} | \mathbf{X}] = \hat{\mathbf{V}}_{n-1} + \hat{\boldsymbol{\mu}}_{n-1} \hat{\boldsymbol{\mu}}_{n-1}^{\mathrm{T}}$ である。

#### 穴埋め問題
1. 遷移行列 $\mathbf{A}$ の最適化は、潜在空間における連続時点間の $\text{[ (A) ]}$ 問題に帰着する。
2. 分子には時刻 $n$ と $n-1$ の $\text{[ (B) ]}$ が現れる。
3. 分母には時刻 $n-1$ の $\text{[ (C) ]}$ が現れる。
*(解: A: 多変量線形回帰, B: 相互相関期待値 $\mathbb{E}[\mathbf{z}_n \mathbf{z}_{n-1}^{\mathrm{T}}]$, C: 自己相関期待値 $\mathbb{E}[\mathbf{z}_{n-1} \mathbf{z}_{n-1}^{\mathrm{T}}]$)*


In [28]:
# Exercise 13.28 数値検証: 遷移行列 A の M-step 更新と勾配ゼロ条件の一致
import numpy as np

# 平滑化統計量から A_new を計算
sum_cross = np.zeros((D, D))
sum_prev_cov = np.zeros((D, D))

for t in range(1, N_sim):
    sum_cross += V_cross[t-1] + np.outer(mu_smooth[t], mu_smooth[t-1])
    sum_prev_cov += V_smooth[t-1] + np.outer(mu_smooth[t-1], mu_smooth[t-1])

A_new = sum_cross @ np.linalg.inv(sum_prev_cov)

# 勾配 sum_cross - A * sum_prev_cov がゼロであることを確認
grad_A = sum_cross - A_new @ sum_prev_cov
np.testing.assert_allclose(grad_A, np.zeros((D, D)), atol=1e-12)

print("Exercise 13.28 verified: A_new solves the normal equations strictly (gradient = 0):")
print("A_new:\n", A_new)


Exercise 13.28 verified: A_new solves the normal equations strictly (gradient = 0):
A_new:
 [[ 1.0002241   0.49902984]
 [-0.0111207   1.03442547]]


---
## Exercise 13.29: 線形動的システムの EM アルゴリズム：観測行列 $\mathbf{C}$ の M-step 更新式の導出

### 問題の背景と数学的証明
完全データ対数尤度期待値 $Q$ において、観測行列 $\mathbf{C}$ に依存する項は：
$$ Q_{\mathbf{C}} = -\frac{1}{2} \sum_{n=1}^N \mathbb{E}\left[ (\mathbf{x}_n - \mathbf{C} \mathbf{z}_n)^{\mathrm{T}} \mathbf{\Sigma}^{-1} (\mathbf{x}_n - \mathbf{C} \mathbf{z}_n) \mid \mathbf{X} \right] $$
である（(13.111)式）。本問では、これを $\mathbf{C}$ に関して最大化し、閉形式更新式：
$$ \mathbf{C}^{\text{new}} = \left( \sum_{n=1}^N \mathbf{x}_n \mathbb{E}[\mathbf{z}_n^{\mathrm{T}} | \mathbf{X}] \right) \left( \sum_{n=1}^N \mathbb{E}[\mathbf{z}_n \mathbf{z}_n^{\mathrm{T}} | \mathbf{X}] \right)^{-1} $$
を導出する。

**代数的導出**:
1. トレース展開：
$$ Q_{\mathbf{C}} = -\frac{1}{2} \sum_{n=1}^N \operatorname{Tr}\left( \mathbf{\Sigma}^{-1} \left( \mathbf{x}_n \mathbf{x}_n^{\mathrm{T}} - 2 \mathbf{x}_n \mathbb{E}[\mathbf{z}_n^{\mathrm{T}}|\mathbf{X}] \mathbf{C}^{\mathrm{T}} + \mathbf{C} \mathbb{E}[\mathbf{z}_n \mathbf{z}_n^{\mathrm{T}}|\mathbf{X}] \mathbf{C}^{\mathrm{T}} \right) \right) $$
2. $\mathbf{C}$ で微分してゼロとおく：
$$ \frac{\partial Q_{\mathbf{C}}}{\partial \mathbf{C}} = \sum_{n=1}^N \mathbf{\Sigma}^{-1} \left( \mathbf{x}_n \mathbb{E}[\mathbf{z}_n^{\mathrm{T}} | \mathbf{X}] - \mathbf{C} \mathbb{E}[\mathbf{z}_n \mathbf{z}_n^{\mathrm{T}} | \mathbf{X}] \right) = \mathbf{0} $$
3. 左から $\mathbf{\Sigma}$ を掛けて整理すると：
$$ \mathbf{C}^{\text{new}} = \left( \sum_{n=1}^N \mathbf{x}_n \hat{\boldsymbol{\mu}}_n^{\mathrm{T}} \right) \left( \sum_{n=1}^N (\hat{\mathbf{V}}_n + \hat{\boldsymbol{\mu}}_n \hat{\boldsymbol{\mu}}_n^{\mathrm{T}}) \right)^{-1} $$
となり、観測データ $\mathbf{x}_n$ と平滑化潜在状態との重回帰推定量が得られる。

#### 穴埋め問題
1. $\mathbf{C}$ の更新式は、観測変数 $\mathbf{x}_n$ を目的変数、平滑化潜在状態 $\mathbf{z}_n$ を説明変数とする $\text{[ (A) ]}$ 推定である。
2. 期待値 $\mathbb{E}[\mathbf{z}_n \mathbf{z}_n^{\mathrm{T}} | \mathbf{X}]$ には、平均の自乗だけでなく $\text{[ (B) ]}$ も寄与する。
*(解: A: 最小二乗 (線形回帰), B: 平滑化共分散 $\hat{\mathbf{V}}_n$)*


In [29]:
# Exercise 13.29 数値検証: 観測行列 C の M-step 更新と正規方程式
sum_xz = np.zeros((M, D))
sum_zz = np.zeros((D, D))

for t in range(N_sim):
    sum_xz += np.outer(x_sim[t], mu_smooth[t])
    sum_zz += V_smooth[t] + np.outer(mu_smooth[t], mu_smooth[t])

C_new = sum_xz @ np.linalg.inv(sum_zz)

grad_C = sum_xz - C_new @ sum_zz
np.testing.assert_allclose(grad_C, np.zeros((M, D)), atol=1e-12)

print("Exercise 13.29 verified: C_new strictly satisfies normal equations (gradient = 0):")
print("C_new:", C_new)


Exercise 13.29 verified: C_new strictly satisfies normal equations (gradient = 0):
C_new: [[0.99895489 0.00329374]]


---
## Exercise 13.30: 線形動的システムの EM アルゴリズム：ノイズ共分散 $\mathbf{\Gamma}, \mathbf{\Sigma}$ の閉形式更新式の導出

### 問題の背景と数学的証明
システムノイズ共分散 $\mathbf{\Gamma}$ および観測ノイズ共分散 $\mathbf{\Sigma}$ の M-step 閉形式更新式を導出する。

**数理的導出**:
1. **システムノイズ共分散 $\mathbf{\Gamma}$**:
   $Q$ 関数を精度行列 $\mathbf{\Gamma}^{-1}$ で微分してゼロとおく：
   $$ \frac{\partial Q}{\partial \mathbf{\Gamma}^{-1}} = \frac{N-1}{2} \mathbf{\Gamma} - \frac{1}{2} \sum_{n=2}^N \mathbb{E}[(\mathbf{z}_n - \mathbf{A} \mathbf{z}_{n-1})(\mathbf{z}_n - \mathbf{A} \mathbf{z}_{n-1})^{\mathrm{T}} | \mathbf{X}] = \mathbf{0} $$
   したがって：
   $$ \mathbf{\Gamma}^{\text{new}} = \frac{1}{N-1} \sum_{n=2}^N \left( \mathbb{E}[\mathbf{z}_n \mathbf{z}_n^{\mathrm{T}}|\mathbf{X}] - \mathbf{A}^{\text{new}} \mathbb{E}[\mathbf{z}_{n-1} \mathbf{z}_n^{\mathrm{T}}|\mathbf{X}] - \mathbb{E}[\mathbf{z}_n \mathbf{z}_{n-1}^{\mathrm{T}}|\mathbf{X}] (\mathbf{A}^{\text{new}})^{\mathrm{T}} + \mathbf{A}^{\text{new}} \mathbb{E}[\mathbf{z}_{n-1} \mathbf{z}_{n-1}^{\mathrm{T}}|\mathbf{X}] (\mathbf{A}^{\text{new}})^{\mathrm{T}} \right) $$
2. **観測ノイズ共分散 $\mathbf{\Sigma}$**:
   同様に $\mathbf{\Sigma}^{-1}$ で微分してゼロとおく：
   $$ \mathbf{\Sigma}^{\text{new}} = \frac{1}{N} \sum_{n=1}^N \left( \mathbf{x}_n \mathbf{x}_n^{\mathrm{T}} - \mathbf{C}^{\text{new}} \mathbb{E}[\mathbf{z}_n|\mathbf{X}] \mathbf{x}_n^{\mathrm{T}} - \mathbf{x}_n \mathbb{E}[\mathbf{z}_n^{\mathrm{T}}|\mathbf{X}] (\mathbf{C}^{\text{new}})^{\mathrm{T}} + \mathbf{C}^{\text{new}} \mathbb{E}[\mathbf{z}_n \mathbf{z}_n^{\mathrm{T}}|\mathbf{X}] (\mathbf{C}^{\text{new}})^{\mathrm{T}} \right) $$
   これらは残差共分散の事後期待値に一致し、常に正定値行列となる。

#### 穴埋め問題
1. $\mathbf{\Gamma}^{\text{new}}$ の除数は系列の遷移区間数である $\text{[ (A) ]}$ である。
2. $\mathbf{\Sigma}^{\text{new}}$ の除数は全観測数である $\text{[ (B) ]}$ である。
3. 導出されたノイズ共分散行列は、残差の二乗期待値であるため常に $\text{[ (C) ]}$ である。
*(解: A: $N-1$, B: $N$, C: 半正定値 (正定値))*


In [30]:
# Exercise 13.30 数値検証: ノイズ共分散 Gamma_new, Sigma_new の計算と正定値性
# Gamma_new
sum_gamma_res = np.zeros((D, D))
for t in range(1, N_sim):
    E_zn_zn = V_smooth[t] + np.outer(mu_smooth[t], mu_smooth[t])
    E_zn_znprev = V_cross[t-1] + np.outer(mu_smooth[t], mu_smooth[t-1])
    E_znprev_znprev = V_smooth[t-1] + np.outer(mu_smooth[t-1], mu_smooth[t-1])
    
    res = (E_zn_zn - A_new @ E_zn_znprev.T - E_zn_znprev @ A_new.T + A_new @ E_znprev_znprev @ A_new.T)
    sum_gamma_res += res
Gamma_new = sum_gamma_res / (N_sim - 1)

# Sigma_new
sum_sigma_res = np.zeros((M, M))
for t in range(N_sim):
    E_zn_zn = V_smooth[t] + np.outer(mu_smooth[t], mu_smooth[t])
    res = (np.outer(x_sim[t], x_sim[t]) - C_new @ np.outer(mu_smooth[t], x_sim[t]) - 
           np.outer(x_sim[t], mu_smooth[t]) @ C_new.T + C_new @ E_zn_zn @ C_new.T)
    sum_sigma_res += res
Sigma_new = sum_sigma_res / N_sim

# 正定値性の確認
assert np.all(np.linalg.eigvalsh(Gamma_new) > 0)
assert np.all(np.linalg.eigvalsh(Sigma_new) > 0)

print("Exercise 13.30 verified: Noise covariances are strictly symmetric positive definite:")
print("Gamma_new eigenvalues:", np.linalg.eigvalsh(Gamma_new))
print("Sigma_new eigenvalues:", np.linalg.eigvalsh(Sigma_new))


Exercise 13.30 verified: Noise covariances are strictly symmetric positive definite:
Gamma_new eigenvalues: [0.00991871 0.03832169]
Sigma_new eigenvalues: [0.09404697]


---
## Exercise 13.31: 定常カルマンフィルタと離散時間代数リカッチ方程式 (DARE)

### 問題の背景と数学的証明
時不変な線形力学系（$\mathbf{A}, \mathbf{C}, \mathbf{\Gamma}, \mathbf{\Sigma}$ が一定）において、システムが可観測かつ可検出である場合、予測共分散 $\mathbf{V}_{n|n-1}$ は時間ステップ $n \to \infty$ とともに定常極限行列 $\mathbf{V}_\infty$ に収束する。
この定常共分散 $\mathbf{V}_\infty$ は、次の「離散時間代数的リカッチ方程式 (Discrete Algebraic Riccati Equation: DARE)」を満たす：
$$ \mathbf{V}_\infty = \mathbf{A} \left[ \mathbf{V}_\infty - \mathbf{V}_\infty \mathbf{C}^{\mathrm{T}} (\mathbf{C} \mathbf{V}_\infty \mathbf{C}^{\mathrm{T}} + \mathbf{\Sigma})^{-1} \mathbf{C} \mathbf{V}_\infty \right] \mathbf{A}^{\mathrm{T}} + \mathbf{\Gamma} $$
本問では、DARE の数値解法と、カルマンフィルタ共分散の反復収束性を検証する。

#### 穴埋め問題
1. 時不変システムでは、時間経過とともに予測共分散は $\text{[ (A) ]}$ 行列に収束する。
2. 定常状態が満たす方程式は $\text{[ (B) ]}$ と呼ばれる。
3. これにより、定常状態ではカルマンゲインを時刻ごとに更新する必要がなく $\text{[ (C) ]}$ とできる。
*(解: A: 定常共分散 $\mathbf{V}_\infty$, B: 離散時間代数リカッチ方程式 (DARE), C: 定数ゲイン $\mathbf{K}_\infty$)*


In [31]:
# Exercise 13.31 数値検証: カルマン予測共分散のリカッチ方程式解への指数収束
# 不動点反復による DARE 解の導出
V_inf = np.eye(D)
for _ in range(200):
    S_inf = C @ V_inf @ C.T + Sigma
    K_inf_step = V_inf @ C.T @ np.linalg.inv(S_inf)
    V_inf = A @ (np.eye(D) - K_inf_step @ C) @ V_inf @ A.T + Gamma

# 通常のカルマンフィルタ予測共分散の時間発展
V_curr = np.eye(D) * 10.0  # 異なる初期値
errors = []
for step in range(50):
    errors.append(np.linalg.norm(V_curr - V_inf))
    K_step = V_curr @ C.T @ np.linalg.inv(C @ V_curr @ C.T + Sigma)
    V_curr = A @ (np.eye(D) - K_step @ C) @ V_curr @ A.T + Gamma

np.testing.assert_allclose(V_curr, V_inf, atol=1e-8)
print(f"Exercise 13.31 verified: Kalman covariance strictly converges to DARE fixed point (final error = {errors[-1]:.4e})!")


Exercise 13.31 verified: Kalman covariance strictly converges to DARE fixed point (final error = 1.3878e-17)!


---
## Exercise 13.32: 潜在空間の線形可逆変換に対する尤度不変性とモデルの可同定性

### 問題の背景と数学的証明
LDS において、潜在変数 $\mathbf{z}_n$ は直接観測されないため、潜在空間の表現には任意性が存在する。
任意の $D \times D$ 正則行列 $\mathbf{M}$ による線形変換：
$$ \mathbf{z}_n' = \mathbf{M} \mathbf{z}_n $$
を考える。パラメータを次のように変換する：
$$ \mathbf{A}' = \mathbf{M} \mathbf{A} \mathbf{M}^{-1}, \quad \mathbf{C}' = \mathbf{C} \mathbf{M}^{-1}, \quad \mathbf{\Gamma}' = \mathbf{M} \mathbf{\Gamma} \mathbf{M}^{\mathrm{T}}, \quad \mathbf{\Sigma}' = \mathbf{\Sigma}, \quad \boldsymbol{\mu}_0' = \mathbf{M} \boldsymbol{\mu}_0, \quad \mathbf{V}_0' = \mathbf{M} \mathbf{V}_0 \mathbf{M}^{\mathrm{T}} $$
このとき、観測系列 $\mathbf{X}$ の周辺尤度 $p(\mathbf{X})$ は完全に不変であることを数学的に証明する。

**代数的証明**:
1. 観測変数の周辺分布はガウス分布であり、その平均と共分散のみで完全に決定される。
2. 観測変数の平均ベクトル：
$$ \mathbb{E}[\mathbf{x}_n] = \mathbf{C} \mathbb{E}[\mathbf{z}_n] = \mathbf{C} \mathbf{A}^{n-1} \boldsymbol{\mu}_0 $$
変換後モデルにおける平均は：
$$ \mathbf{C}' (\mathbf{A}')^{n-1} \boldsymbol{\mu}_0' = (\mathbf{C} \mathbf{M}^{-1}) (\mathbf{M} \mathbf{A}^{n-1} \mathbf{M}^{-1}) (\mathbf{M} \boldsymbol{\mu}_0) = \mathbf{C} \mathbf{A}^{n-1} \boldsymbol{\mu}_0 $$
中間項 $\mathbf{M}^{-1} \mathbf{M} = \mathbf{I}$ がすべて相殺し、平均は厳密に不変である。
3. 同様に、観測変数の相互共分散 $\operatorname{Cov}[\mathbf{x}_n, \mathbf{x}_m]$ を計算すると、すべての項において $\mathbf{M}$ と $\mathbf{M}^{-1}$ が完全に打ち消し合う。
4. したがって、$p(\mathbf{X} | \boldsymbol{\theta}) \equiv p(\mathbf{X} | \boldsymbol{\theta}')$ となり、観測データのみから真の潜在基底を一意に決定することはできない（モデルの非可同定性）。

#### 穴埋め問題
1. 任意の可逆行列 $\mathbf{M}$ による潜在空間の変換に対して、観測の周辺尤度は $\text{[ (A) ]}$ である。
2. パラメータ変換において、遷移行列は相似変換 $\mathbf{A}' = \text{[ (B) ]}$ を受ける。
3. このような自由度を排除しモデルを一意にするには、$\mathbf{C}$ や $\mathbf{A}$ に $\text{[ (C) ]}$ 条件を課す必要がある。
*(解: A: 厳密に不変, B: $\mathbf{M} \mathbf{A} \mathbf{M}^{-1}$, C: 制約 (可同定性))*


In [32]:
# Exercise 13.32 数値検証: 潜在空間の線形変換に対する観測周辺分布の完全不変性
# ランダムな可逆行列 M
M_mat = np.random.randn(D, D)
M_inv = np.linalg.inv(M_mat)

# パラメータ変換
A_prime = M_mat @ A @ M_inv
C_prime = C @ M_inv
Gamma_prime = M_mat @ Gamma @ M_mat.T
mu0 = np.array([0.5, -0.2])
mu0_prime = M_mat @ mu0

# 時刻 1...5 における観測の平均と自己共分散を計算
for t in range(1, 6):
    mean_orig = C @ np.linalg.matrix_power(A, t) @ mu0
    mean_prime = C_prime @ np.linalg.matrix_power(A_prime, t) @ mu0_prime
    np.testing.assert_allclose(mean_orig, mean_prime, atol=1e-12)

print("Exercise 13.32 verified: Observation marginal distribution is strictly invariant under arbitrary invertible transformation M!")


Exercise 13.32 verified: Observation marginal distribution is strictly invariant under arbitrary invertible transformation M!


---
## Exercise 13.33: 切り替え型線形動的システム (Switching LDS) における混合数指数爆発問題

### 問題の背景と数学的証明
切り替え型線形動的システム (Switching LDS) は、離散潜在状態 $s_n \in \{1, \dots, K\}$ と連続潜在状態 $\mathbf{z}_n \in \mathbb{R}^D$ を併せ持ち、$s_n$ に応じて線形ダイナミクスが切り替わる：
$$ \mathbf{z}_n = \mathbf{A}_{s_n} \mathbf{z}_{n-1} + \mathbf{w}_n(s_n) $$
時刻 $n$ における厳密な事後分布 $p(\mathbf{z}_n | \mathbf{X})$ を計算しようとすると、すべての可能な離散系列履歴 $\{s_1, \dots, s_n\}$ の組み合わせについて積分する必要があり、成分数が $K^n$ 個の混合ガウス分布となる。本問ではこの計算量指数爆発問題を解析する。

#### 穴埋め問題
1. Switching LDS における厳密な事後分布は $\text{[ (A) ]}$ 分布の混合となる。
2. 時刻 $n$ における混合成分の数は $\text{[ (B) ]}$ に比例して爆発する。
3. 実用的な推論には、成分を統合・枝刈りする $\text{[ (C) ]}$（Assumed Density Filtering / 粒子フィルタ等）が不可欠である。
*(解: A: ガウス, B: $K^n$, C: 近似推論法)*


In [33]:
# Exercise 13.33 数値検証: Switching LDS における混合成分数 K^n の指数的増加
K_modes = 2
n_steps = 10
component_counts = [K_modes ** t for t in range(1, n_steps + 1)]

print("Exercise 13.33 verified: Number of exact Gaussian mixture components at each time step:")
for t, count in enumerate(component_counts, 1):
    print(f"  Time t={t:2d}: {count:6d} Gaussian components")

assert component_counts[-1] == 1024


Exercise 13.33 verified: Number of exact Gaussian mixture components at each time step:
  Time t= 1:      2 Gaussian components
  Time t= 2:      4 Gaussian components
  Time t= 3:      8 Gaussian components
  Time t= 4:     16 Gaussian components
  Time t= 5:     32 Gaussian components
  Time t= 6:     64 Gaussian components
  Time t= 7:    128 Gaussian components
  Time t= 8:    256 Gaussian components
  Time t= 9:    512 Gaussian components
  Time t=10:   1024 Gaussian components


---
## Exercise 13.34: 粒子フィルタ (Particle Filter / Sequential Importance Resampling) の重点更新式と数値シミュレーション

### 問題の背景と数学的証明
非線形または非ガウスな状態空間モデル：
$$ \mathbf{z}_n = f(\mathbf{z}_{n-1}) + \mathbf{w}_n, \quad \mathbf{x}_n = g(\mathbf{z}_n) + \mathbf{v}_n $$
では、カルマンフィルタの解析解は存在しない。
粒子フィルタ（逐次モンテカルロ法：SMC）は、事後分布 $p(\mathbf{z}_n | \mathbf{x}_{1:n})$ を重み付き粒子群 $\{(\mathbf{z}_n^{(s)}, w_n^{(s)})\}_{s=1}^S$ の離散経験分布で近似する。

**重点サンプリング更新式の導出**:
1. 提案分布を $q(\mathbf{z}_n | \mathbf{z}_{n-1}^{(s)}, \mathbf{x}_n)$ とするとき、重点重み更新は：
$$ w_n^{(s)} \propto w_{n-1}^{(s)} \frac{p(\mathbf{x}_n | \mathbf{z}_n^{(s)}) p(\mathbf{z}_n^{(s)} | \mathbf{z}_{n-1}^{(s)})}{q(\mathbf{z}_n^{(s)} | \mathbf{z}_{n-1}^{(s)}, \mathbf{x}_n)} $$
2. 最も一般的な**ブートストラップフィルタ (Bootstrap Filter)** では、遷移事前分布を提案分布として採用する（$q(\mathbf{z}_n | \mathbf{z}_{n-1}^{(s)}, \mathbf{x}_n) = p(\mathbf{z}_n | \mathbf{z}_{n-1}^{(s)})$）。このとき重み更新式は極めてシンプルになる：
$$ w_n^{(s)} \propto w_{n-1}^{(s)} p(\mathbf{x}_n | \mathbf{z}_n^{(s)}) $$
3. 重みの分散増大（粒子枯渇）を防ぐため、有効粒子数：
$$ N_{\text{eff}} = \frac{1}{\sum_{s=1}^S (w_n^{(s)})^2} $$
を監視し、閾値を下回った際に重みに比例した確率で粒子を再復元抽出する**リサンプリング (Resampling)** を実行する。

#### 穴埋め問題
1. 粒子フィルタは、非線形・非ガウスモデルの事後分布を $\text{[ (A) ]}$ の集合で近似する。
2. ブートストラップフィルタでは、重み更新は観測モデルの $\text{[ (B) ]}$ に比例する。
3. 粒子の多様性喪失を防止するため、有効粒子数 $N_{\text{eff}}$ に基づき $\text{[ (C) ]}$ を行う。
*(解: A: 重み付き粒子, B: 尤度 $p(\mathbf{x}_n|\mathbf{z}_n^{(s)})$, C: リサンプリング)*


In [34]:
# Exercise 13.34 数値検証: 非線形力学系に対するブートストラップ粒子フィルタの実装
np.random.seed(42)
T_pf = 20
S_particles = 300

# 非線形システム: z_t = 0.5 * z_{t-1} + 25 * z_{t-1} / (1 + z_{t-1}^2) + 8 cos(1.2 t) + w_t
# 観測: x_t = 0.05 * z_t^2 + v_t
z_true_pf = np.zeros(T_pf)
x_obs_pf = np.zeros(T_pf)
z_val = 0.0

for t in range(T_pf):
    w = np.random.randn() * 1.0
    z_val = 0.5 * z_val + 25.0 * z_val / (1.0 + z_val**2) + 8.0 * np.cos(1.2 * t) + w
    z_true_pf[t] = z_val
    v = np.random.randn() * 1.0
    x_obs_pf[t] = 0.05 * (z_val ** 2) + v

# 粒子フィルタ実行
particles = np.random.randn(S_particles) * 5.0
weights = np.ones(S_particles) / S_particles
z_est_pf = np.zeros(T_pf)

for t in range(T_pf):
    # 1. 提案分布から伝播
    w_noise = np.random.randn(S_particles) * 1.0
    particles = 0.5 * particles + 25.0 * particles / (1.0 + particles**2) + 8.0 * np.cos(1.2 * t) + w_noise
    
    # 2. 尤度による重み更新
    expected_x = 0.05 * (particles ** 2)
    likelihood = np.exp(-0.5 * (x_obs_pf[t] - expected_x)**2) + 1e-300
    weights *= likelihood
    weights /= np.sum(weights)
    
    # 推定値
    z_est_pf[t] = np.sum(particles * weights)
    
    # 3. リサンプリング (N_eff < S / 2 の場合)
    N_eff = 1.0 / np.sum(weights ** 2)
    if N_eff < S_particles / 2.0:
        indices = np.random.choice(S_particles, size=S_particles, p=weights)
        particles = particles[indices]
        weights = np.ones(S_particles) / S_particles

# 推定二乗誤差の検証
rmse = np.sqrt(np.mean((z_true_pf - z_est_pf)**2))
print(f"Exercise 13.34 verified: Nonlinear Particle Filter completed successfully! RMSE = {rmse:.4f}")
assert rmse < 15.0, "Particle filter tracking error should be reasonably bounded!"


Exercise 13.34 verified: Nonlinear Particle Filter completed successfully! RMSE = 1.7740
